<p class="lead">This notebook is the R edition of the sixth unit of the course. The lecture slides are a map; this text is the territory. Units 2–4 built classifiers whose shape was fixed in advance — a straight boundary (logistic regression), a Gaussian per class (naive Bayes), axis-parallel splits (trees) — and Unit 5 found structure without labels. This unit introduces the family of models that today dominates the field: <em>artificial neural networks</em>, which assemble many simple units into a function flexible enough to represent almost any relationship between inputs and outputs, and which learn that function by gradient descent. The unit follows the official subunits in order: the biological inspiration (Section 2), what a network is (3), what it is good and bad at (4), where it is used in medicine (5), the computational model of the neuron (6), the simple perceptron (7), the multilayer perceptron and backpropagation (8), training in practice (9), and a complete worked example on the breast-cancer data (10). Every algorithm is first written on paper, then implemented in a few lines of base R, and only then compared with the <code>nnet</code> and <code>neuralnet</code> packages. No deep-learning framework is used; everything that happens inside the network is visible.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Describe the human brain in numbers (neurons, synapses, speed, parallelism) and explain which of its properties artificial networks borrow and which they do not.</li>
<li>Identify the components of a neural network — inputs, weights, bias, summation, activation function, output, layers — in a diagram or a formula, and count its parameters.</li>
<li>Compute the output of a McCulloch–Pitts neuron by hand for the AND and OR functions, and prove geometrically why no single neuron computes XOR.</li>
<li>State Rosenblatt's learning rule and the perceptron convergence theorem, trace the algorithm by hand for one epoch, and explain why it fails on non-separable data.</li>
<li>Write forward propagation of a multilayer perceptron as matrix products, derive backpropagation with the chain rule for a 2-2-1 network, and compute every partial derivative by hand for one example.</li>
<li>Explain the role of the learning rate, the number of epochs, early stopping, weight initialisation, input standardisation and regularisation, and recognise divergence, overfitting and vanishing gradients in a loss curve.</li>
<li>Train, select and evaluate a multilayer perceptron in R with <code>nnet</code> using the methodology of Supplement S1, and compare it fairly with the methods of Units 3–4.</li>
<li>Explain what "deep" adds to a multilayer perceptron, and what convolutional, recurrent and attention-based networks change, as preparation for Unit 8.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets the output. Exercises (green boxes) ask you to modify the chunk above them and re-run; they are the fastest way to learn. Blue boxes carry key ideas, orange boxes carry common misconceptions, and purple boxes connect this unit to other units or to the bioinformatics course. The notebook uses only base R with <code>stats</code>, <code>MASS</code>, <code>nnet</code>, <code>neuralnet</code>, <code>pROC</code>, <code>mlbench</code>, <code>e1071</code>, <code>rpart</code>, <code>class</code> and <code>randomForest</code>; in a fresh Google Colab R runtime a student runs once <code>install.packages(c("nnet","neuralnet","pROC","mlbench","e1071","rpart","randomForest"))</code>, and no data is downloaded. Three algorithms are traced by hand before the code confirms them: the neuron computing AND and OR (Section 6.3), the perceptron learning rule (Section 7.3) and backpropagation (Section 8.4). Evaluation methodology — held-out sets, cross-validation, sensitivity and specificity, ROC curves — is assumed from Supplement S1.</p></div>



In [ ]:
# Setup: libraries used throughout. Seeds are fixed so that every quoted number is reproducible.
suppressPackageStartupMessages({
  library(nnet); library(neuralnet); library(mlbench); library(MASS); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"; PURPLE <- "#6a3d9a"
sigmoid <- plogis                       # R's plogis(z) = 1 / (1 + exp(-z)) is the logistic sigmoid
cat("R", as.character(getRversion()), "| nnet", as.character(packageVersion("nnet")),
    "| neuralnet", as.character(packageVersion("neuralnet")), "\n")



The printout records the R and package versions these notes were built with. In R the logistic sigmoid already exists as `plogis()` (and its inverse, the logit, as `qlogis()`), so we alias it to `sigmoid` and never implement \( 1/(1+e^{-z}) \) by hand — a first example of statistics being built into the language.

## 1. Introduction

Every classifier of this course so far has had a shape chosen by its designer. Logistic regression (Unit 2) draws a straight boundary; Gaussian naive Bayes (Unit 3) assumes each class is a bell; a decision tree (Unit 4) cuts feature space into boxes; a support vector machine with a radial kernel bends the boundary, but only in the way the kernel permits. When the true relationship between features and label has a different shape, the method's <em>bias</em> — the error it makes even with infinite data — cannot be reduced by collecting more data. Artificial neural networks approach the problem from the opposite direction: assemble a large number of very simple units, each computing a weighted sum followed by a fixed non-linear function, connect them in layers, and let the training data decide the shape of the function. The result is a model that can, with enough units, approximate essentially any continuous relationship (Section 4), and whose parameters — the <em>weights</em> of the connections — are all learned by the same procedure, gradient descent on a loss function.

Unit 1 placed this family under the <strong>connectionist</strong> approach to artificial intelligence: knowledge is stored implicitly, distributed over the weights, and acquired by training rather than programmed. The approach is old. McCulloch and Pitts proposed the artificial neuron in 1943,<sup class="c"><a href="#r1">[1]</a></sup> Rosenblatt built the first learning machine, the perceptron, in 1958,<sup class="c"><a href="#r2">[2]</a></sup> Minsky and Papert proved its limits in 1969,<sup class="c"><a href="#r3">[3]</a></sup> and Rumelhart, Hinton and Williams popularised the algorithm that trains multi-layer networks, backpropagation, in 1986.<sup class="c"><a href="#r4">[4]</a></sup> Yet for most of that time neural networks were one method among many, often outperformed by the support vector machines and random forests of Units 3–4 on the small tabular datasets that were available. The turning point was 2012, when a network with eight layers trained on a million labelled images reduced the error of the ImageNet recognition benchmark by a margin that no other method had approached.<sup class="c"><a href="#r5">[5]</a></sup> The ingredients were the same as in 1986; what changed was the amount of data and of computation, and a handful of engineering details (Section 9) that made deep networks trainable in practice.<sup class="c"><a href="#r6">[6]</a></sup> Since then neural networks have become the default method wherever the input is an image, a signal, a sequence or a text — that is, wherever the features cannot be designed by hand.

This unit stops before "deep" learning proper, which Unit 8 applies to medical images. Its subject is the <strong>multilayer perceptron</strong> (MLP): the simplest network with hidden layers, and the component out of which every modern architecture is built. Understanding how an MLP computes its output, how backpropagation computes the gradient, and what can go wrong during training is the whole of what is needed to read the deep-learning literature critically.



In [ ]:
# Figure 1: a single neuron, a one-layer perceptron and a multilayer perceptron, drawn as graphs.
draw_net <- function(sizes, labels, title, components = FALSE) {
  plot.new(); plot.window(xlim = c(0, 1), ylim = c(0, 1.1)); title(main = title, cex.main = 1)
  xs <- seq(0.12, 0.88, length.out = length(sizes)); pos <- list()
  for (li in seq_along(sizes)) {
    n <- sizes[li]; ys <- 0.5 + 0.12 * seq(-(n - 1) / 2, (n - 1) / 2, length.out = n)
    pos[[li]] <- cbind(xs[li], ys)
    text(xs[li], max(ys) + 0.13, labels[li], cex = 0.8)
  }
  for (li in seq_len(length(sizes) - 1)) for (a in seq_len(sizes[li])) for (b in seq_len(sizes[li + 1]))
    segments(pos[[li]][a, 1], pos[[li]][a, 2], pos[[li + 1]][b, 1], pos[[li + 1]][b, 2], col = GREY, lwd = 0.7)
  for (li in seq_along(sizes)) {
    fill <- if (li == 1) "white" else if (li < length(sizes)) BLUE else ORANGE
    points(pos[[li]], pch = 21, bg = fill, cex = 3.2, lwd = 1.3)
  }
  if (components) text(0.5, 0.06, expression(z == sum(w[i] * x[i]) + b ~ "," ~ a == phi(z)), cex = 0.9)
}
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
draw_net(c(3, 1), c("inputs x1..x3", "output a"), "(a) a single neuron", components = TRUE)
draw_net(c(4, 2), c("input layer", "output layer"), "(b) single-layer perceptron")
draw_net(c(3, 4, 2), c("input", "hidden", "output"), "(c) multilayer perceptron 3-4-2")



<p class="caption"><b>Figure 1.</b> The three objects of this unit. (a) A single artificial neuron: inputs \(x_i\) arrive along connections carrying weights \(w_i\); the unit computes the weighted sum \(z = \sum_i w_i x_i + b\) (the bias \(b\) is a weight on a constant input of 1) and applies an activation function \(\varphi\) to produce the output \(a\). (b) A single-layer perceptron: several neurons sharing the same inputs; the input layer performs no computation and is not counted as a layer of neurons. (c) A multilayer perceptron with one hidden layer (blue) and an output layer (orange); every unit of one layer is connected to every unit of the next, hence <em>fully connected</em> or <em>dense</em>. Network (c) has \(3 \times 4 + 4 + 4 \times 2 + 2 = 26\) parameters.</p>

## 2. General characteristics of the human brain

The name "neural network" is a claim of kinship with the brain, and the claim should be examined before it is used. This section summarises what is known about the organ, states which of its properties the artificial networks of this unit imitate, and is explicit about which they do not.

### 2.1 The brain in numbers

The adult human brain contains about 86 billion neurons — \(86.1 \pm 8.1 \times 10^9\) in the isotropic fractionator counts of Azevedo and colleagues, together with a roughly equal number of non-neuronal (glial) cells.<sup class="c"><a href="#r7">[7]</a></sup> The popular figure of 100 billion is an older estimate that the direct counts did not confirm; and, contrary to another popular statement, the human brain is not exceptional for its size but is a linearly scaled-up primate brain, remarkable mainly for the number of neurons it packs into the cerebral cortex (about 16 billion).<sup class="c"><a href="#r8">[8]</a></sup> Each cortical neuron receives on the order of thousands of synapses; stereological counts in the human neocortex give about \(1.5 \times 10^{14}\) synapses in that structure alone,<sup class="c"><a href="#r9">[9]</a></sup> so the usual textbook range of \(10^{14}\)–\(10^{15}\) synapses for the whole brain is the right order of magnitude.

### 2.2 The neuron and the action potential

A neuron has three functional parts. The <strong>dendrites</strong> are branched extensions that receive signals from other neurons; the <strong>soma</strong> (cell body) integrates them; the <strong>axon</strong> is a single long fibre that carries the output signal to other cells, ending in terminals that form <strong>synapses</strong> with the dendrites of the next neurons. Communication across a synapse is chemical, excitatory synapses pushing the receiving membrane potential up and inhibitory ones down, and the <em>strength</em> of each synapse — how much one incoming signal moves the potential — is the biological quantity to which the weights of an artificial network correspond.

The output signal is the <strong>action potential</strong> (spike): when the membrane potential at the start of the axon exceeds a threshold of about \(-55\) to \(-50\) mV, voltage-gated sodium channels open, the potential swings to about \(+30\) mV and back within a millisecond or two, and the pulse propagates down the axon without loss of amplitude. Hodgkin and Huxley described the ionic mechanism quantitatively in 1952, in the work for which they received the Nobel Prize.<sup class="c"><a href="#r10">[10]</a></sup> The action potential is <em>all-or-none</em>: its size does not depend on how far the threshold was exceeded. Information about the strength of the input is carried instead by the <em>rate</em> of firing — a neuron driven harder fires more spikes per second, up to a limit set by the refractory period of a few milliseconds during which it cannot fire again. This <strong>rate code</strong> is the biological reading of the activation function of Section 6: the output of an artificial unit is a smooth number that stands for a firing rate, not for an individual spike. The chunk below simulates the simplest model that captures the mechanism, the <em>leaky integrate-and-fire</em> neuron proposed by Lapicque in 1907,<sup class="c"><a href="#r11">[11]</a></sup> and plots its firing rate as a function of the input current.



In [ ]:
# A leaky integrate-and-fire neuron: the membrane potential integrates the input current, leaks back to rest,
# and emits a spike (then resets) on reaching threshold. Firing rate versus input is the f-I curve.
tau_m <- 20; R_m <- 10; V_rest <- -70; V_th <- -50; V_reset <- -70; t_ref <- 2   # ms, MOhm, mV
dt <- 0.1; Tdur <- 500
lif <- function(I) {                       # returns potential trace and spike times for input current I (nA)
  V <- V_rest; refr <- 0; trace <- numeric(Tdur / dt); spikes <- c()
  for (s in seq_len(Tdur / dt)) {
    if (refr > 0) { refr <- refr - dt; V <- V_reset }
    else V <- V + dt / tau_m * (-(V - V_rest) + R_m * I)
    if (V >= V_th) { spikes <- c(spikes, s * dt); V <- 30; refr <- t_ref }
    trace[s] <- V
  }
  list(trace = trace, spikes = spikes)
}
currents <- seq(0, 50, by = 1)
rates <- sapply(currents, function(I) length(lif(I)$spikes) / Tdur * 1000)    # spikes per second (Hz)
for (I in c(1.5, 2.5, 4, 8, 50)) cat(sprintf("input %4.1f nA -> firing rate %5.0f Hz\n", I, length(lif(I)$spikes) / Tdur * 1000))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
tt <- seq(0, Tdur - dt, by = dt)
plot(tt[1:1500], lif(2.5)$trace[1:1500], type = "l", col = BLUE, lwd = 1, xlab = "time (ms)",
     ylab = "membrane potential (mV)", main = "All-or-none spikes; rate tracks input", ylim = c(-75, 35))
lines(tt[1:1500], lif(4)$trace[1:1500], col = ORANGE, lwd = 1)
abline(h = V_th, col = GREY, lty = 2); legend("topright", c("2.5 nA", "4 nA"), col = c(BLUE, ORANGE), lwd = 1, bty = "n", cex = 0.8)
plot(currents, rates, type = "o", pch = 19, cex = 0.5, col = BLUE, xlab = "input current (nA)",
     ylab = "firing rate (Hz)", main = "f-I curve: threshold, then saturation")



<p class="caption"><b>Figure 2.</b> Left: membrane potential of a leaky integrate-and-fire neuron for two input currents. Every spike has the same amplitude; a stronger input only makes them more frequent. Right: firing rate as a function of input current. Below about 2 nA the leak wins and the neuron never reaches threshold (rate 0); above it the rate rises steeply and then bends towards the ceiling \(1/t_{\text{ref}} = 500\) Hz imposed by the refractory period. The curve has exactly the shape of the activation functions of Section 6: a threshold, a steep rise, a plateau — a biological sigmoid.</p>

### 2.3 Slow, parallel, distributed, robust, plastic

Five properties of the brain as a computing device are the ones that connectionist models set out to capture.

**It is slow but massively parallel.** A neuron fires at most a few hundred times per second and a signal crosses a synapse in about a millisecond, some \(10^6\) times slower than a transistor switches. Yet a person recognises a face in about 100–150 ms — a hundred serial steps at most. The brain compensates for slow components by having \(10^{11}\) of them working at once; whatever algorithm it runs must be shallow and wide, not deep and serial. Artificial networks inherit this: each layer is computed in parallel as one matrix product, which is why graphics processors (Section 11) suit them.

**Knowledge is distributed.** No single neuron stores the concept "grandmother"; a concept is a pattern of activity over many neurons, and a neuron takes part in many patterns. In an artificial network, likewise, no weight means anything by itself; a trained network's knowledge is spread over all of its weights, which is the root of its opacity (Section 4).

**It degrades gracefully.** Neurons die daily throughout adult life, and the brain tolerates the loss of a sizeable fraction before function is noticeably impaired; damage produces a gradual decline rather than a crash. Section 4.3 shows the same property in an artificial network from which weights are deleted at random.

**It learns by changing synapses.** The strength of a synapse changes with use. Hebb proposed the rule in 1949: when a presynaptic neuron repeatedly takes part in firing a postsynaptic one, the connection between them is strengthened — in the slogan, <em>cells that fire together wire together</em>.<sup class="c"><a href="#r12">[12]</a></sup> Hebb's rule is the ancestor of every learning rule in this unit, and the chunk below implements it in three lines.

**It builds its own features.** The visual cortex is a hierarchy: early areas respond to oriented edges, later ones to contours, textures, parts and finally objects and faces. Nobody designs these features; they emerge from exposure to the visual world. The deep networks of Unit 8 are the artificial counterpart, and their learned first-layer filters resemble the edge detectors of primary visual cortex closely enough that the resemblance is one of the field's standard illustrations.<sup class="c"><a href="#r6">[6]</a></sup>



In [ ]:
# Hebb's rule: dw_i = eta * x_i * y. The weight of an input that fires WITH the output grows;
# the weight of an input that fires independently of the output barely grows.
set.seed(0); steps <- 500
post     <- runif(steps) < 0.3                                       # output fires on ~30% of steps
pre_corr <- ifelse(post, runif(steps) < 0.9, runif(steps) < 0.05)    # input 1 usually fires WITH the output
pre_rand <- runif(steps) < 0.3                                       # input 2 fires independently
eta <- 0.01; w <- c(0, 0)
for (s in seq_len(steps)) w <- w + eta * c(pre_corr[s], pre_rand[s]) * post[s]
tab <- data.frame(input = c("input 1 (correlated)", "input 2 (independent)"),
                  P_fires = round(c(mean(pre_corr), mean(pre_rand)), 3),
                  P_fires_given_output = round(c(mean(pre_corr[post]), mean(pre_rand[post])), 3),
                  final_weight = round(w, 3))
print(tab, row.names = FALSE)



Both inputs fire about 30 % of the time, so a rule that strengthened a synapse whenever the <em>input</em> was active would treat them alike. Hebb's rule is conditional on the coincidence of input and output, and the input that co-fires with the output in about 90 % of the output's spikes ends with a weight two to three times larger than the input that co-fires only by chance. The network has learned a <em>correlation</em> from experience without a teacher — the mechanism that the perceptron rule of Section 7 turns into supervised learning by adding an error signal.

### 2.4 Brain and computer, side by side

| Property | Human brain | Digital computer |
|---|---|---|
| Basic element | neuron (~\(10^{11}\)) with ~\(10^3\)–\(10^4\) synapses each | transistor (~\(10^{10}\) per chip) |
| Switching time | ~1 ms | ~0.1 ns |
| Operation | analogue, noisy, massively parallel | digital, exact, mostly serial |
| Memory | distributed in synaptic strengths; content-addressable | localised at addresses; address-based |
| Programming | learned from experience; synaptic plasticity | written explicitly by a programmer |
| Fault tolerance | graceful degradation; daily neuron loss tolerated | a single bit error can crash a program |
| Energy | ~20 W | hundreds of watts for a training GPU |

<div class="box warn"><h4>Inspired by, not a model of</h4><p>Artificial neural networks borrow from the brain the ideas of simple units, weighted connections, layers, distributed representation and learning by weight adjustment — and nothing else. The units of this unit have no spikes, no time, no dendritic computation, no neurotransmitters, no glia; their learning rule, backpropagation, requires each synapse to know the error at the output and to be adjusted by the same global signal, for which no biological mechanism is known; and their training needs millions of labelled examples where a child needs a handful. A neural network is a statistical model fitted by optimisation, and the vocabulary of "neurons" and "learning" is a historical metaphor. Statements such as "the network thinks" or "the algorithm learns like a brain" should be read with that in mind.<sup class="c"><a href="#r6">[6]</a></sup></p></div>

## 3. What an artificial neural network is

### 3.1 Units, weights, layers, architecture

An <strong>artificial neural network</strong> is a directed graph whose nodes are <em>units</em> (neurons) and whose edges are <em>connections</em>, each carrying a real-valued <em>weight</em>. Every unit computes the same thing: the weighted sum of its inputs plus a <em>bias</em>, passed through an <em>activation function</em> (Section 6). Units are organised into <strong>layers</strong>: an <em>input layer</em>, which merely holds the feature vector and performs no computation; zero or more <em>hidden layers</em>, so called because their values are neither given as data nor read as predictions; and an <em>output layer</em>, whose values are the prediction. The <strong>architecture</strong> is the specification of how many layers there are, how many units each has, which activation each uses and how the layers are connected. The notation 9-16-1 means 9 inputs, one hidden layer of 16 units, one output; by convention the number of layers of a network counts the layers that compute, so a 9-16-1 network is a <em>two-layer</em> network with <em>one hidden layer</em>. Textbooks differ on this count — some include the input layer — so when it matters, state the architecture explicitly.

Two kinds of architecture exist. In a <strong>feed-forward</strong> network the graph has no cycles: information flows from the input layer through the hidden layers to the output, and the output is a function of the current input alone. All the networks of this unit are feed-forward. In a <strong>recurrent</strong> network some connections loop back, so that the output at one time step depends on the inputs at earlier steps; recurrent networks process sequences — speech, electrocardiograms, text — and Section 11 returns to them.

### 3.2 The network as a parametrised function

Mathematically, a feed-forward network with parameters \(\boldsymbol{\theta}\) (all its weights and biases) is a function \(f(\mathbf{x}; \boldsymbol{\theta})\) from the feature vector \(\mathbf{x} \in \mathbb{R}^p\) to an output \(\hat{\mathbf{y}}\). The architecture fixes the <em>form</em> of \(f\); training chooses the <em>values</em> of \(\boldsymbol{\theta}\) that make \(f(\mathbf{x}_i; \boldsymbol{\theta}) \approx y_i\) on the training sample, by minimising a loss function with gradient descent — exactly the recipe of Unit 2 for linear and logistic regression, with a more flexible \(f\). A network is a model, in the vocabulary of Unit 1, and backpropagation is the algorithm that fits it.

The number of parameters is the first thing to compute when reading an architecture, because it bounds the amount of data the network needs. For a fully connected network with layer sizes \(n_0, n_1, \dots, n_L\) (with \(n_0 = p\)), each layer \(l\) has an \(n_{l-1} \times n_l\) weight matrix and \(n_l\) biases, so

<div class="eq">\[ \#\boldsymbol{\theta} = \sum_{l=1}^{L} \left( n_{l-1}\, n_l + n_l \right). \]</div>



In [ ]:
# Counting parameters from the architecture: weights plus biases of every computing layer.
n_parameters <- function(sizes) sum(head(sizes, -1) * tail(sizes, -1) + tail(sizes, -1))
archs <- list("single neuron, 2 inputs" = c(2, 1),
              "single neuron, 9 inputs (breast cancer)" = c(9, 1),
              "2-2-1 (the XOR network, Section 8)" = c(2, 2, 1),
              "9-16-1 (breast cancer, Section 8)" = c(9, 16, 1),
              "64-16-5 (synthetic glyphs, Section 9)" = c(64, 16, 5),
              "784-500-500-10 (handwritten digits at 28x28)" = c(784, 500, 500, 10))
data.frame(layer_sizes = sapply(archs, paste, collapse = "-"),
           parameters  = sapply(archs, n_parameters), row.names = names(archs))



A single neuron with 9 inputs has 10 parameters — the same number as a logistic regression on 9 features, which is not a coincidence (next subsection). Adding one hidden layer of 16 units multiplies the count by about 16; a modest three-layer network for 28 × 28 images already has about 650,000 parameters, more than the number of training images in most medical datasets. The ratio of parameters to training cases is the quantity to keep in view throughout Sections 8–10: a network with more parameters than cases can memorise the training sample perfectly, and only the methodology of Supplement S1 reveals whether it has learned anything else.

### 3.3 A single neuron with a sigmoid is logistic regression

The simplest network — one unit, sigmoid activation — computes \(\hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b)\) with \(\sigma(z) = 1/(1+e^{-z})\). That is exactly the logistic-regression model of Unit 2: the weights are the regression coefficients, the bias is the intercept, and if the unit is trained by gradient descent on the cross-entropy loss, it must arrive at the same solution, because the loss is convex and has a single minimum. The chunk below trains the neuron by hand on two standardised features of the breast-cancer data<sup class="c"><a href="#r13">[13]</a></sup> and compares its weights with R's `glm` fitted as a logistic regression.



In [ ]:
# Wisconsin breast cancer from mlbench (the R stand-in for sklearn's load_breast_cancer).
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                             # drop the 16 rows with missing values
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # 9 ordinal factors -> numeric
y_mal <- as.numeric(bc$Class == "malignant")               # 1 = malignant, 0 = benign
feat2 <- c("Cell.size", "Bare.nuclei"); Z2 <- scale(as.matrix(bc[, feat2]))

w <- c(0, 0); b <- 0                                        # a sigmoid neuron by full-batch gradient descent
for (it in 1:30000) { p <- sigmoid(Z2 %*% w + b); w <- w - 1.0 * t(Z2) %*% (p - y_mal) / length(y_mal); b <- b - mean(p - y_mal) }
g <- glm(y_mal ~ Z2, family = binomial)                    # the same model fitted by R's glm (IRLS)
cat(sprintf("sigmoid neuron : w = (%.3f, %.3f)  b = %.3f\n", w[1], w[2], b))
cat(sprintf("glm (logistic) : w = (%.3f, %.3f)  b = %.3f\n", coef(g)[2], coef(g)[3], coef(g)[1]))
cat(sprintf("identical predictions on the %d cases: %.3f\n", nrow(Z2),
            mean((sigmoid(Z2 %*% w + b) >= 0.5) == (predict(g, type = "response") >= 0.5))))



The weights agree to three decimals, and the two models make the same prediction for every case: the hand-coded neuron and `glm` are fitting the same convex model, reaching the same minimum by different numerical routes (gradient descent versus iteratively reweighted least squares). Everything you know about logistic regression — reading a coefficient, standardising features, the straight boundary — therefore applies to a single neuron. More important for this unit, a network with hidden layers is a logistic regression <em>on features that the network has learned</em>: the output unit of the 9-16-1 network of Section 8 is a logistic regression on 16 inputs, but those 16 inputs are not measured features; they are non-linear combinations of the 9 measurements, chosen by training to make the final logistic regression accurate. This is what "learning features" means, and it is the single most important idea of the unit.

<div class="box key"><h4>Key idea</h4><p>A neural network is a parametrised function \(f(\mathbf{x}; \boldsymbol{\theta})\) whose output layer is a familiar model (logistic regression for classification, linear regression for regression) and whose hidden layers compute the features that model receives. Training adjusts all the parameters at once by gradient descent, so the features are learned for the task rather than designed by the analyst.</p></div>

## 4. Advantages and limitations

### 4.1 What networks do well

**Non-linearity and universal approximation.** With a non-linear activation function in the hidden layer, an MLP can represent functions no straight line, Gaussian or box can. The result has a precise form: Cybenko (1989) proved that a network with one hidden layer of sigmoid units and a linear output can approximate any continuous function on a bounded domain to any desired accuracy, given enough hidden units;<sup class="c"><a href="#r14">[14]</a></sup> Hornik, Stinchcombe and White extended the result to any non-constant, bounded, continuous activation.<sup class="c"><a href="#r15">[15]</a></sup> Two cautions are essential. The theorem guarantees <em>existence</em>, not <em>learnability</em>: it does not say how many units are needed (the number can be astronomically large), nor that gradient descent will find the right weights, nor that a network that fits the training data will generalise. And "one hidden layer suffices" is a statement about representation, not about efficiency; Section 11 explains why depth helps in practice. The chunk below shows the theorem at work on a synthetic signal, using `nnet` with a linear output.

**Learned features.** Section 3.3 made the point: the hidden layers are feature extractors trained for the task. For tabular data with a handful of designed features this matters little; for images, signals and sequences, where no one knows how to design the features, it is decisive.<sup class="c"><a href="#r6">[6]</a></sup>

**Parallelism and graceful degradation.** Every unit of a layer computes independently, so a layer is one matrix product, and a network with millions of parameters runs in milliseconds on parallel hardware. And because knowledge is distributed over many weights, no single weight is critical: deleting a fraction of them degrades performance gradually rather than abruptly (Section 4.3), a property that no decision tree — in which cutting one node removes a whole sub-tree — shares.



In [ ]:
# Universal approximation: a one-hidden-layer tanh/sigmoid network (nnet) fits a synthetic ECG beat
# (P wave, QRS complex, T wave as Gaussian bumps). More hidden units -> a better fit, up to a point.
ecg_beat <- function(t) {
  g <- function(mu, s, a) a * exp(-((t - mu) / s)^2)
  g(0.20, 0.045, 0.15) + g(0.49, 0.015, -0.10) + g(0.52, 0.020, 1.0) + g(0.555, 0.015, -0.25) + g(0.78, 0.06, 0.30)
}
t <- seq(0, 1, length.out = 300); signal <- ecg_beat(t); Xt <- matrix((t - 0.5) * 6, ncol = 1)
par(mfrow = c(1, 4), mar = c(4, 3.2, 2.5, 1))
for (h in c(2, 4, 16, 64)) {
  set.seed(0)
  net <- nnet(Xt, signal, size = h, linout = TRUE, maxit = 2000, trace = FALSE, MaxNWts = 5000, abstol = 1e-8)
  pred <- predict(net, Xt); rmse <- sqrt(mean((pred - signal)^2))
  cat(sprintf("%3d sigmoid hidden units: %4d parameters, RMSE = %.4f\n", h, n_parameters(c(1, h, 1)), rmse))
  plot(t, signal, type = "l", col = GREY, lwd = 3, xlab = "time (s)", ylab = if (h == 2) "amplitude (mV)" else "",
       main = sprintf("%d hidden units\nRMSE %.3f", h, rmse)); lines(t, pred, col = BLUE, lwd = 1.4)
}



<p class="caption"><b>Figure 3.</b> A one-hidden-layer network (<code>nnet</code>, logistic hidden units, linear output) fitted to a synthetic ECG beat with 2, 4, 16 and 64 units. Two units can place only two smooth steps and miss the QRS complex entirely; four capture the R wave crudely; sixteen reproduce the whole beat with a small error, and sixty-four smaller still. The fit is not guaranteed to improve with width, however: <code>nnet</code>'s BFGS optimiser can stop in a local minimum of a non-convex loss, so a wider network sometimes fits no better than a narrower one, which illustrates the second limitation of Section 4.2 — the theorem promises that the weights exist, not that training will find them.</p>

### 4.2 What networks do badly

**They are data-hungry.** The flexibility that lets a network represent any function also lets it fit noise. A network with \(10^5\) parameters trained on \(10^2\) patients will memorise them; the methods of Section 9 (early stopping, regularisation) mitigate but do not remove the problem. On small tabular datasets — the typical clinical cohort of a few hundred cases — random forests and support vector machines (Units 3–4) frequently match or beat networks, and Section 10 tests this honestly on the breast-cancer data.

**They are opaque.** A trained tree is a readable rule; a trained network is a few thousand numbers whose individual meaning is nil. Post-hoc methods — permutation importance (Section 10), saliency maps (Unit 8) — help, but none gives an explanation a clinician can verify against physiology, and the regulatory frameworks of Unit 1 increasingly require one.

**They have many hyperparameters.** Architecture, activation function, learning rate or optimiser, weight decay, number of epochs, initialisation — each is a choice with no theory to guide it, and each interacts with the others. Section 9 goes through them; in practice they are chosen by cross-validation (Supplement S1), at a cost in computing time that the simpler methods do not incur.

**Training is non-convex.** The loss of logistic regression has one minimum; the loss of a network with hidden layers has many local minima, saddle points and flat plateaus, so the result of training depends on the starting weights and the order of the data (hence `set.seed`), and two runs can give different models of different quality. Figure 3 showed a small instance.

**They are brittle outside the training distribution.** A network extrapolates with confidence into regions where it has seen no data; the shortcut-learning and dataset-shift failures of Unit 1 are, in practice, failures of networks deployed beyond their training data.

### 4.3 Graceful degradation, measured

The chunk below trains networks on a set of simulated 8 × 8 glyphs (a stand-in for medical images; R has no `digits` dataset, so we generate five noisy templates), then deletes an increasing fraction of their first-layer weights at random — the artificial counterpart of losing neurons — and measures the accuracy that remains. Two widths are compared. The `nnet` weight vector begins with the first-layer weights, so zeroing a random fraction of its leading entries and re-evaluating with `maxit = 0` performs the deletion.



In [ ]:
# Simulated 8x8 glyphs for five classes: a template per class plus Gaussian noise (the R stand-in for load_digits).
set.seed(0)
templates <- list(
  vbar = { m <- matrix(0, 8, 8); m[, 4:5] <- 1; m },
  hbar = { m <- matrix(0, 8, 8); m[4:5, ] <- 1; m },
  diag = { m <- matrix(0, 8, 8); for (i in 1:8) m[i, i] <- 1; for (i in 1:7) m[i, i + 1] <- 1; m },
  box  = { m <- matrix(0, 8, 8); m[2:7, 2] <- 1; m[2:7, 7] <- 1; m[2, 2:7] <- 1; m[7, 2:7] <- 1; m },
  blob = { m <- matrix(0, 8, 8); m[3:6, 3:6] <- 1; m })
make_digits <- function(n_per = 150, noise = 0.9) {
  X <- do.call(rbind, lapply(templates, function(tp)
    t(replicate(n_per, pmax(0, pmin(1, as.vector(tp) + rnorm(64, 0, noise)))))))
  list(X = X, y = factor(rep(0:4, each = n_per)))
}
dig <- make_digits(); cat("simulated glyphs:", nrow(dig$X), "images x", ncol(dig$X), "pixels,",
                          nlevels(dig$y), "classes\n")
set.seed(1); tr_d <- sample(nrow(dig$X), round(0.7 * nrow(dig$X)))
Zd <- scale(dig$X); Zd_tr <- Zd[tr_d, ]; Zd_te <- Zd[-tr_d, ]; yd_tr <- dig$y[tr_d]; yd_te <- dig$y[-tr_d]
Ytr <- class.ind(yd_tr)

# Delete a random fraction of the trained first-layer weights and measure held-out accuracy.
fractions <- seq(0, 0.9, by = 0.05); set.seed(0); results <- list()
for (h in c(8, 64)) {
  set.seed(0); net <- nnet(Zd_tr, Ytr, size = h, softmax = TRUE, maxit = 300, trace = FALSE, MaxNWts = 8000)
  n_first <- h * (ncol(Zd_tr) + 1)                        # first-layer weights are the leading entries of net$wts
  accs <- sapply(fractions, function(fr) {
    w <- net$wts; kill <- sample(n_first, round(fr * n_first)); w[kill] <- 0
    net0 <- nnet(Zd_tr, Ytr, size = h, softmax = TRUE, Wts = w, maxit = 0, trace = FALSE, MaxNWts = 8000)
    mean((max.col(predict(net0, Zd_te)) - 1) == (as.integer(yd_te) - 1))
  })
  results[[as.character(h)]] <- accs
  cat(sprintf("%2d hidden units: intact %.3f | 30%% deleted %.3f | 60%% deleted %.3f\n", h, accs[1], accs[7], accs[13]))
}
plot(fractions * 100, results[["64"]], type = "o", pch = 19, cex = 0.6, col = BLUE, ylim = c(0, 1),
     xlab = "first-layer weights deleted (%)", ylab = "test accuracy", main = "Graceful degradation of a trained network")
lines(fractions * 100, results[["8"]], type = "o", pch = 19, cex = 0.6, col = ORANGE)
abline(h = 0.2, col = GREY, lty = 3); text(5, 0.24, "chance (5 classes)", col = GREY, cex = 0.8)
legend("bottomleft", c("64 hidden units", "8 hidden units"), col = c(BLUE, ORANGE), lwd = 1, pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 4.</b> Test accuracy of two trained networks as a random fraction of their first-layer weights is set to zero. The decline is gradual: the wider network keeps most of its accuracy with a large fraction of its connections removed, and even the narrow one tolerates substantial loss before accuracy falls towards chance. Redundancy in the distributed representation absorbs the damage. Compare a program, in which a single corrupted instruction halts execution, or a decision tree, in which deleting the root node removes every decision.</p>

<div class="box key"><h4>Exercise 1</h4><p>In the chunk above, delete weights of the <em>output</em> layer (the trailing entries of <code>net$wts</code>) instead of the first. The output layer has far fewer weights; is the network equally tolerant to losing them? Then repeat the experiment with an <code>rpart</code> tree from Unit 4 by pruning random internal nodes, and compare the shape of the curve.</p></div>

## 5. Applications in medicine

Neural networks entered medicine long before deep learning. Baker and colleagues trained a three-layer network in 1995 to predict breast-cancer malignancy from the standardised mammographic descriptors of the BI-RADS lexicon, reporting an area under the ROC curve higher than that of the attending radiologists on the same cases — an early instance of the pattern that the course has stressed: a network fitted to features designed by clinicians, evaluated with the ROC methodology of Supplement S1.<sup class="c"><a href="#r16">[16]</a></sup> By 2006 the literature was large enough for Lisboa and Taktak to review it systematically for cancer decision support: of 396 studies, only 27 were clinical trials or randomised controlled trials, and among those 21 showed an increase in benefit to healthcare provision, 6 showed none, and none showed a decrease — a cautiously positive verdict and a reminder that most published models were never evaluated in a way that could show benefit.<sup class="c"><a href="#r17">[17]</a></sup>

The deep-learning era produced the systems that Unit 1 cited as evidence of specialist-level performance, all of them multilayer networks of the kind this unit describes, scaled up and given convolutional layers for images (Unit 8):

| Year | Task | Input | Architecture | Reported performance | Ref. |
|---|---|---|---|---|---|
| 2016 | Diabetic retinopathy | retinal fundus photographs | deep convolutional network | sensitivity 97.5 %, specificity 93.4 % at the high-sensitivity operating point (EyePACS-1) | <sup class="c"><a href="#r18">[18]</a></sup> |
| 2017 | Skin-cancer classification | dermoscopic and clinical photographs | deep convolutional network | on par with 21 dermatologists on biopsy-proven images | <sup class="c"><a href="#r19">[19]</a></sup> |
| 2018 | Referral decisions from retinal OCT | 3-D optical coherence tomography | two networks: segmentation then classification | error rate 5.5 %, matching retina specialists on 997 scans | <sup class="c"><a href="#r20">[20]</a></sup> |
| 2019 | Arrhythmia detection | single-lead ambulatory ECG | 34-layer convolutional network | F1 0.837, exceeding the average cardiologist on 12 rhythm classes | <sup class="c"><a href="#r21">[21]</a></sup> |

Three lessons belong in this unit. First, every one of these systems is a classifier of the kind built in Section 10 — inputs, hidden layers, softmax output, cross-entropy loss, gradient descent — differing from the 9-16-1 network in scale and in the use of convolution. Second, their inputs are raw pixels or voltages: the <em>learned-feature</em> advantage of Section 4.1 is what made them possible, and why the breast-cancer data, with nine hand-designed features, do not favour networks (Section 10). Third, as Rajkomar, Dean and Kohane stressed in their review for clinicians, performance on a curated test set is the beginning of the evidence, not the end: prospective validation, calibration, dataset shift and integration into workflow decide whether a model helps patients.<sup class="c"><a href="#r22">[22]</a></sup>

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>The breakthrough that students of the bioinformatics course know best, AlphaFold, is a neural network: its inputs are multiple sequence alignments and residue-pair features, and its output is a protein structure. QSAR models of molecular activity — molecular descriptors as the input layer, activity as the output — were among the earliest applications of multilayer perceptrons in the life sciences, and the cross-validation methodology used there is the one Section 10 applies.</p></div>

## 6. The computational model of the neuron

### 6.1 From McCulloch–Pitts to the modern unit

McCulloch and Pitts abstracted the neuron of Section 2.2 into a logical device: binary inputs, each either excitatory or inhibitory, a threshold, and a binary output that fires when the number of active excitatory inputs reaches the threshold and no inhibitory input is active. They showed that networks of such units can compute any Boolean function, and thereby that the brain's logic could in principle be realised by neurons.<sup class="c"><a href="#r1">[1]</a></sup> Their unit did not learn. Rosenblatt's perceptron (Section 7) gave each input a real-valued <em>weight</em> adjustable from data, and the modern unit generalises the threshold to an arbitrary <em>activation function</em>. The computation of a unit with \(p\) inputs is in two steps:

<div class="eq">\[ z = \sum_{i=1}^{p} w_i x_i + b = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \varphi(z), \]
<span class="where">where \(z\) is the <em>pre-activation</em> (net input), \(w_i\) the weight of input \(i\), \(b\) the bias, \(\varphi\) the activation function and \(a\) the <em>activation</em> (output) of the unit. The bias can be absorbed into the weights by appending a constant input \(x_0 = 1\) with weight \(w_0 = b\), which is how it is often drawn and how the code of Section 7 treats it.</span></div>

Geometrically, \(z = 0\) is a hyperplane in feature space — a line when \(p = 2\) — and the sign of \(z\) says on which side of it \(\mathbf{x}\) lies. A single unit therefore divides feature space into two half-spaces, whatever activation it uses; the activation only determines how the output varies with the distance from the boundary.

### 6.2 Activation functions

The activation function is what makes a network more than a linear model: without it, a stack of layers computes \(\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2\), which is again a linear function of \(\mathbf{x}\), and no number of layers would add anything. Five functions cover almost all practice.

| Function | Formula \(\varphi(z)\) | Derivative \(\varphi'(z)\) | Range | Use |
|---|---|---|---|---|
| **Step** (Heaviside) | \(1\) if \(z \geq 0\), else \(0\) | \(0\) everywhere (undefined at 0) | \(\{0, 1\}\) | McCulloch–Pitts unit, perceptron; not trainable by gradient |
| **Sigmoid** (logistic) | \(\sigma(z) = \dfrac{1}{1 + e^{-z}}\) | \(\sigma(z)\,(1 - \sigma(z))\) | \((0, 1)\) | output unit for binary classification; also hidden units (what <code>nnet</code> uses) |
| **Hyperbolic tangent** | \(\tanh(z)\) | \(1 - \tanh^2(z)\) | \((-1, 1)\) | hidden units; zero-centred, which eases training compared with the sigmoid |
| **ReLU** (rectified linear unit) | \(\max(0, z)\) | \(1\) if \(z > 0\), else \(0\) | \([0, \infty)\) | default hidden unit since about 2011; cheap, and its derivative does not vanish for \(z > 0\) |
| **Softmax** | \(\dfrac{e^{z_k}}{\sum_j e^{z_j}}\) for each output \(k\) | (a matrix; see Section 8.2) | each in \((0,1)\), sum 1 | output layer for \(K\)-class classification |

The derivative column is as important as the function column, because training (Section 8) multiplies derivatives of activations along the path from the output back to each weight. The sigmoid's derivative has a maximum of \(0.25\) at \(z = 0\) and is essentially zero for \(|z| > 5\): a unit whose pre-activation is large in magnitude is <em>saturated</em>, and the gradient through it vanishes — the mechanism behind the vanishing-gradient problem of Section 9.6. The ReLU, introduced to deep networks by Nair and Hinton,<sup class="c"><a href="#r26">[26]</a></sup> has derivative exactly 1 for every positive input, so gradients pass through active units undiminished; its price is that a unit with \(z < 0\) for every training case has zero gradient and never recovers (a "dead" unit). The step function has derivative zero everywhere, which is why the perceptron (Section 7) needs a learning rule that is not gradient descent, and why the sigmoid replaced it as soon as gradient-based training was wanted. R's `nnet` uses logistic hidden units; the ReLU appears in the from-scratch network of Section 8.6.



In [ ]:
# The four scalar activation functions of Section 6.2 and their derivatives.
z <- seq(-6, 6, length.out = 601)
acts <- list(step = list(f = as.numeric(z >= 0), d = rep(0, length(z))),
             sigmoid = list(f = sigmoid(z), d = sigmoid(z) * (1 - sigmoid(z))),
             tanh = list(f = tanh(z), d = 1 - tanh(z)^2),
             ReLU = list(f = pmax(z, 0), d = as.numeric(z > 0)))
par(mfrow = c(2, 4), mar = c(3.4, 3.4, 2, 1))
for (nm in names(acts)) { plot(z, acts[[nm]]$f, type = "l", col = BLUE, lwd = 1.6, main = nm, xlab = "", ylab = if (nm == "step") "phi(z)" else ""); abline(h = 0, v = 0, col = GREY, lwd = 0.5) }
for (nm in names(acts)) { plot(z, acts[[nm]]$d, type = "l", col = ORANGE, lwd = 1.6, main = "", xlab = "z", ylab = if (nm == "step") "phi'(z)" else "", ylim = c(-0.1, 1.1)); abline(h = 0, v = 0, col = GREY, lwd = 0.5) }

# Softmax turns K logits into a probability distribution over K classes.
softmax <- function(v) { e <- exp(v - max(v)); e / sum(e) }
logits <- c(2.0, 1.0, 0.1)
cat("logits", logits, "-> softmax", round(softmax(logits), 3), "| sum =", round(sum(softmax(logits)), 6), "\n")
cat("logits + 5 -> softmax", round(softmax(logits + 5), 3), "(adding a constant changes nothing)\n")
cat("logits x 3 -> softmax", round(softmax(logits * 3), 3), "(scaling sharpens the distribution)\n")



<p class="caption"><b>Figure 5.</b> Top row: the step, sigmoid, tanh and ReLU activation functions. Bottom row: their derivatives. The step has zero derivative everywhere, so no gradient can flow through it. The sigmoid's derivative peaks at 0.25 and is negligible beyond \(|z| \approx 5\) (saturation); tanh's peaks at 1 but saturates similarly. The ReLU's derivative is 1 for every positive input — the property that made deep networks trainable — and 0 for negative inputs. Below the figure, the softmax of three logits: the outputs are positive, sum to one, are unchanged by adding a constant to every logit, and become sharper when the logits are scaled up. With \(K = 2\) classes the softmax reduces to the sigmoid of the difference of the two logits.</p>

### 6.3 Worked example by hand: one neuron computes AND and OR, and cannot compute XOR

Take a McCulloch–Pitts unit with two binary inputs and the step activation, \(a = \mathbb{1}[w_1 x_1 + w_2 x_2 + b \geq 0]\). With weights \(w_1 = w_2 = 1\) and bias \(b = -1.5\) the unit computes the logical **AND**; with the same weights and \(b = -0.5\) it computes **OR**:

| \(x_1\) | \(x_2\) | \(z_{\text{AND}} = x_1 + x_2 - 1.5\) | AND | \(z_{\text{OR}} = x_1 + x_2 - 0.5\) | OR | XOR (target) |
|---|---|---|---|---|---|---|
| 0 | 0 | \(-1.5\) | 0 | \(-0.5\) | 0 | 0 |
| 0 | 1 | \(-0.5\) | 0 | \(0.5\) | 1 | 1 |
| 1 | 0 | \(-0.5\) | 0 | \(0.5\) | 1 | 1 |
| 1 | 1 | \(0.5\) | 1 | \(1.5\) | 1 | 0 |

Geometrically, the boundary \(z = 0\) is the line \(x_1 + x_2 = 1.5\) for AND — it separates the corner \((1,1)\) from the other three — and the line \(x_1 + x_2 = 0.5\) for OR, which separates \((0,0)\) from the other three. NOT is computed by one input with \(w = -1\), \(b = 0.5\); NAND by \(w_1 = w_2 = -1\), \(b = 1.5\).

**Why no single unit computes XOR.** Suppose weights \(w_1, w_2, b\) existed with \(a = 1\) exactly for \((0,1)\) and \((1,0)\). The four cases require

<div class="eq">\[ b < 0 \quad (0,0 \mapsto 0), \qquad w_2 + b \geq 0 \quad (0,1 \mapsto 1), \qquad w_1 + b \geq 0 \quad (1,0 \mapsto 1), \qquad w_1 + w_2 + b < 0 \quad (1,1 \mapsto 0). \]</div>

Adding the second and third inequalities gives \(w_1 + w_2 + 2b \geq 0\), and subtracting the first (\(-b > 0\)) gives \(w_1 + w_2 + b > 0\) — which contradicts the fourth. No weights exist. In the picture, XOR asks a single straight line to put the two diagonal corners \((0,1)\) and \((1,0)\) on one side and \((0,0)\) and \((1,1)\) on the other, and no line does: the two classes are not <strong>linearly separable</strong>. This is the observation that Minsky and Papert turned into a general theory of what single-layer perceptrons cannot do,<sup class="c"><a href="#r3">[3]</a></sup> and the reason hidden layers are necessary: XOR is \((x_1 \text{ OR } x_2) \text{ AND NOT } (x_1 \text{ AND } x_2)\), a composition of three linearly separable functions, which a network with one hidden layer of two units computes exactly (Section 8.5 trains one). The chunk confirms the table and searches exhaustively for XOR weights on a grid.



In [ ]:
# A McCulloch-Pitts unit computing AND, OR, NAND, and an exhaustive search showing no weights compute XOR.
unit <- function(X, w, b) as.integer(X %*% w + b >= 0)
Xb <- rbind(c(0, 0), c(0, 1), c(1, 0), c(1, 1)); xor_target <- c(0, 1, 1, 0)
truth <- data.frame(x1 = Xb[, 1], x2 = Xb[, 2],
                    AND = unit(Xb, c(1, 1), -1.5), OR = unit(Xb, c(1, 1), -0.5),
                    NAND = unit(Xb, c(-1, -1), 1.5), XOR_target = xor_target)
print(truth, row.names = FALSE)
grid <- seq(-3, 3, length.out = 121); best <- 0
for (w1 in grid) for (w2 in grid) {
  z <- Xb %*% c(w1, w2)                                        # pre-activation without bias, length 4
  preds <- outer(grid, as.vector(z), function(b, zz) as.integer(zz + b >= 0))   # rows = candidate biases
  best <- max(best, max(rowSums(sweep(preds, 2, xor_target, `==`))))
}
cat(sprintf("\nXOR: best number of correct cases over all %d weight triples = %d of 4\n", length(grid)^3, best))
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
for (cfg in list(list("AND", truth$AND, 1.5), list("OR", truth$OR, 0.5), list("XOR", xor_target, NA))) {
  nm <- cfg[[1]]; tgt <- cfg[[2]]; ln <- cfg[[3]]
  plot(Xb[, 1], Xb[, 2], pch = ifelse(tgt == 1, 15, 19), col = ifelse(tgt == 1, BLUE, ORANGE), cex = 2.2,
       xlim = c(-0.4, 1.4), ylim = c(-0.4, 1.4), xlab = "x1", ylab = "x2", main = nm)
  if (!is.na(ln)) abline(a = ln, b = -1, lty = 2) else text(0.5, 0.5, "no line\nseparates", col = GREY, cex = 0.9)
}



<p class="caption"><b>Figure 6.</b> The four input patterns of two binary variables, coloured by the output of AND (left), OR (centre) and XOR (right). For AND and OR the dashed line \(z = 0\) of the hand-worked unit separates the two outputs. For XOR the exhaustive search over more than 1.7 million weight triples finds at most 3 of 4 cases correct, confirming the algebraic proof: the two classes occupy the diagonals of the square and no line separates them.</p>

<div class="box key"><h4>Exercise 2</h4><p>Find by hand weights and bias for NOR and for the function "\(x_1\) AND NOT \(x_2\)", check them with <code>unit()</code>, and draw their boundaries. Then compute XOR as a two-step network on paper: hidden unit \(h_1 = \text{OR}(x_1, x_2)\), hidden unit \(h_2 = \text{NAND}(x_1, x_2)\), output \(= \text{AND}(h_1, h_2)\). Write out the table of \(h_1, h_2\) and the output for the four inputs. Section 8.5 will learn these weights from data instead of designing them.</p></div>

## 7. The simple perceptron

### 7.1 Rosenblatt's machine

Rosenblatt's perceptron of 1958 was the first machine that learned to classify from examples.<sup class="c"><a href="#r2">[2]</a></sup> The Mark I Perceptron took its input from a 20 × 20 grid of photocells (<em>sensory units</em>), connected them through fixed random wiring to <em>association units</em>, and those through adjustable weights — motor-driven potentiometers — to <em>response units</em>; only the last set of connections learned. In modern terms the perceptron is a single-layer network of threshold units, trained by the rule of the next subsection; the "simple perceptron" of the syllabus is one such unit, computing \(\hat{y} = \text{sign}(\mathbf{w}^\top \mathbf{x} + b)\) with classes coded \(y \in \{-1, +1\}\).

### 7.2 The learning rule and the convergence theorem

The rule is a supervised version of Hebb's: adjust the weights only when the unit is wrong, and in the direction that would have made it right. For a training example \((\mathbf{x}, y)\) with prediction \(\hat{y}\),

<div class="eq">\[ \text{if } \hat{y} \neq y: \qquad \mathbf{w} \leftarrow \mathbf{w} + \eta\, y\, \mathbf{x}, \qquad b \leftarrow b + \eta\, y, \]
<span class="where">where \(\eta > 0\) is the learning rate. If the prediction is correct, nothing changes.</span></div>

The rule has a transparent derivation. A mistake on a positive example (\(y = +1\), but \(z = \mathbf{w}^\top\mathbf{x} + b < 0\)) means \(z\) was too small; after the update the pre-activation of the same example becomes \(z + \eta(\lVert \mathbf{x} \rVert^2 + 1)\), which is larger, so the example has moved towards the correct side. A mistake on a negative example moves \(z\) down by the same amount. Each update is a step of size \(\eta\) in the direction of \(y\mathbf{x}\), which is the negative gradient of the <em>perceptron loss</em> \(\max(0, -y z)\) — the rule is stochastic gradient descent on a loss that is zero for correctly classified examples and grows linearly with the violation for misclassified ones. The algorithm cycles through the training sample (one pass is an <strong>epoch</strong>) until an epoch produces no update.

Does it stop? Rosenblatt claimed so; Novikoff gave the proof in 1962.<sup class="c"><a href="#r23">[23]</a></sup> <strong>Perceptron convergence theorem.</strong> If the training sample is linearly separable with margin \(\gamma\) — that is, some unit-norm weight vector \(\mathbf{w}^*\) (bias absorbed) satisfies \(y_i\, \mathbf{w}^{*\top} \mathbf{x}_i \geq \gamma > 0\) for every \(i\) — and every example satisfies \(\lVert \mathbf{x}_i \rVert \leq R\), then the perceptron rule started from \(\mathbf{w} = \mathbf{0}\) makes at most \((R/\gamma)^2\) mistakes before it classifies every example correctly, whatever the order of presentation and for any \(\eta\). The bound depends neither on the number of examples nor on the dimension. The theorem's premise is also its limitation: if the sample is not separable, the algorithm never stops (Section 7.4).

### 7.3 Worked example by hand: one epoch of the perceptron on AND

Code the AND problem with \(y \in \{-1, +1\}\): the inputs \((0,0), (0,1), (1,0)\) have \(y = -1\) and \((1,1)\) has \(y = +1\). Take \(\eta = 1\), start from \(\mathbf{w} = (0, 0)\), \(b = 0\), predict \(\hat{y} = +1\) when \(z \geq 0\) and \(-1\) otherwise, and present the four examples in the order above. The first epoch:

| step | \(\mathbf{x}\) | \(y\) | \(z = \mathbf{w}^\top\mathbf{x} + b\) | \(\hat{y}\) | mistake? | update | \(\mathbf{w}\) after | \(b\) after |
|---|---|---|---|---|---|---|---|---|
| 1 | \((0,0)\) | \(-1\) | \(0\) | \(+1\) | yes | \(b \mathrel{+}= -1\) | \((0, 0)\) | \(-1\) |
| 2 | \((0,1)\) | \(-1\) | \(-1\) | \(-1\) | no | — | \((0, 0)\) | \(-1\) |
| 3 | \((1,0)\) | \(-1\) | \(-1\) | \(-1\) | no | — | \((0, 0)\) | \(-1\) |
| 4 | \((1,1)\) | \(+1\) | \(-1\) | \(-1\) | yes | \(\mathbf{w} \mathrel{+}= (1,1)\), \(b \mathrel{+}= 1\) | \((1, 1)\) | \(0\) |

After one epoch, \(\mathbf{w} = (1, 1)\) and \(b = 0\): the line \(x_1 + x_2 = 0\) passes through the origin, which is still wrong for \((0,0)\) (\(z = 0 \Rightarrow \hat{y} = +1\)). The second epoch starts with a mistake on \((0,0)\) (bias to \(-1\)), then on \((0,1)\) (\(z = 0\); \(\mathbf{w} \leftarrow (1, 0)\), \(b \leftarrow -2\)), and so on. The chunk below runs the same arithmetic to convergence; it should reproduce the four rows above exactly and end at a separating line.



In [ ]:
# The perceptron learning rule in base R, on the AND problem, with every step logged (compare with the hand table).
perceptron_fit <- function(X, y, eta = 1, max_epochs = 20, log = FALSE) {
  w <- rep(0, ncol(X)); b <- 0; mistakes <- integer(0); rows <- list()
  for (epoch in 1:max_epochs) {
    m <- 0
    for (i in seq_len(nrow(X))) {
      z <- sum(w * X[i, ]) + b; yhat <- if (z >= 0) 1 else -1
      miss <- yhat != y[i]
      if (miss) { w <- w + eta * y[i] * X[i, ]; b <- b + eta * y[i]; m <- m + 1 }
      if (log) rows[[length(rows) + 1]] <- data.frame(epoch, x1 = X[i, 1], x2 = X[i, 2], y = y[i],
                 z = z, yhat = yhat, mistake = miss, w1 = w[1], w2 = w[2], b = b)
    }
    mistakes <- c(mistakes, m); if (m == 0) break
  }
  list(w = w, b = b, mistakes = mistakes, log = if (log) do.call(rbind, rows) else NULL)
}
X_and <- Xb; y_and <- c(-1, -1, -1, 1)
fit_and <- perceptron_fit(X_and, y_and, log = TRUE)
print(fit_and$log[fit_and$log$epoch == 1, ], row.names = FALSE)                 # epoch 1: the hand-traced rows
cat(sprintf("\nmistakes per epoch: %s -> converged after %d epochs, %d updates total\n",
            paste(fit_and$mistakes, collapse = " "), length(fit_and$mistakes), sum(fit_and$mistakes)))
cat(sprintf("final w = (%g, %g), b = %g\n", fit_and$w[1], fit_and$w[2], fit_and$b))



The first four rows match the hand table line by line. The algorithm needs six epochs — five with mistakes and a sixth that confirms there are none — and eleven updates in total, ending with \(\mathbf{w} = (2, 1)\), \(b = -3\): the line \(2x_1 + x_2 = 3\), which passes exactly through \((1,1)\) (classified \(+1\) because \(z = 0 \geq 0\)) and leaves the other three corners on the negative side. It is <em>a</em> separating line, not the "best" one; the perceptron stops at the first solution it finds, and a different presentation order would yield a different line.

<div class="box warn"><h4>Common misconception</h4><p>"The perceptron minimises the number of errors." It does not; it stops at the first weight vector with zero training errors, if one exists, and otherwise never stops. The support vector machine of Unit 3 solves the problem the perceptron leaves open — <em>which</em> separating line — by choosing the one with the largest margin, and logistic regression (Unit 2) and the Adaline of Section 7.5 minimise smooth losses that are defined whether or not the data are separable.</p></div>

### 7.4 The perceptron on real data: breast cancer, and the failure on XOR

The next chunk applies the rule to the two standardised features of Section 3.3 for all 683 cases, and draws the decision line after selected epochs. The two classes overlap, so the sample is not linearly separable and the theorem does not apply; the figure shows what happens then.



In [ ]:
# The perceptron on two breast-cancer features (standardised). The classes overlap: the boundary keeps moving.
y_pm <- ifelse(y_mal == 1, 1, -1)                                    # +1 = malignant, -1 = benign
snapshots <- c(1, 2, 3, 5, 10, 20, 50, 100); w <- c(0, 0); b <- 0; states <- list()
for (epoch in 1:100) {
  for (i in seq_len(nrow(Z2))) if (y_pm[i] * (sum(w * Z2[i, ]) + b) <= 0) { w <- w + y_pm[i] * Z2[i, ]; b <- b + y_pm[i] }
  if (epoch %in% snapshots) states[[as.character(epoch)]] <- list(w = w, b = b,
      acc = mean(ifelse(Z2 %*% w + b >= 0, 1, -1) == y_pm))
}
cat("training accuracy after epoch:",
    paste(sprintf("%d:%.3f", snapshots, sapply(as.character(snapshots), function(e) states[[e]]$acc)), collapse = "  "), "\n")
par(mfrow = c(2, 4), mar = c(3.6, 3.6, 2, 1)); xs <- c(-2.5, 4.5)
for (e in snapshots) {
  st <- states[[as.character(e)]]
  plot(Z2[, 1], Z2[, 2], col = ifelse(y_pm == 1, ORANGE, BLUE), pch = 19, cex = 0.3, xlim = c(-2.5, 4.5),
       ylim = c(-2, 4.5), xlab = "cell size (std.)", ylab = "bare nuclei (std.)", main = sprintf("epoch %d: acc %.3f", e, st$acc))
  lines(xs, -(st$w[1] * xs + st$b) / st$w[2], lwd = 1.3)
}



<p class="caption"><b>Figure 7.</b> The perceptron's decision line after 1, 2, 3, 5, 10, 20, 50 and 100 epochs on the two-feature breast-cancer problem. Training accuracy is already high after the first epoch and then <em>fluctuates</em> rather than improves, because every epoch contains misclassified cases that move the line; the algorithm never terminates, and the line at epoch 100 is not systematically better than the one at epoch 5. On non-separable data the perceptron's final weights depend on when one chooses to stop — an arbitrary choice that the methods of Section 7.5 and Unit 2 remove by minimising a loss instead of chasing zero errors.</p>



In [ ]:
# Mistakes per epoch: the perceptron converges on AND (separable) and never on XOR (not separable).
X_xor <- Xb; y_xor <- c(-1, 1, 1, -1)
hist_and <- perceptron_fit(X_and, y_and, max_epochs = 30)$mistakes
hist_xor <- perceptron_fit(X_xor, y_xor, max_epochs = 30)$mistakes
cat("AND: mistakes per epoch", hist_and, "\n")
cat("XOR: mistakes per epoch", hist_xor, sprintf("-> minimum %d per epoch, never 0\n", min(hist_xor)))
plot(seq_along(hist_and), hist_and, type = "o", pch = 19, col = GREEN, ylim = c(0, 4), xlab = "epoch",
     ylab = "mistakes in the epoch", main = "Perceptron convergence theorem in action")
lines(seq_along(hist_xor), hist_xor, type = "o", pch = 15, col = ORANGE)
legend("topright", c("AND (separable)", "XOR (not separable)"), col = c(GREEN, ORANGE), lwd = 1, pch = c(19, 15), bty = "n", cex = 0.8)



<p class="caption"><b>Figure 8.</b> Mistakes per epoch of the perceptron rule on AND and on XOR. On AND the count reaches zero at the sixth epoch and the algorithm stops, as the convergence theorem guarantees. On XOR the count never reaches zero within the 30 epochs allowed — nor would it in 30 million — because no separating line exists; the weights cycle through a finite set of states for ever. This is the experiment behind Minsky and Papert's critique.<sup class="c"><a href="#r3">[3]</a></sup></p>

### 7.5 Adaline, the delta rule, and the link to gradient descent

Two years after Rosenblatt, Widrow and Hoff proposed a unit that differs from the perceptron in one respect with large consequences.<sup class="c"><a href="#r24">[24]</a></sup> The <strong>Adaline</strong> (adaptive linear neuron) compares the target not with the thresholded output \(\hat{y} = \text{sign}(z)\) but with the <em>linear</em> output \(z = \mathbf{w}^\top\mathbf{x} + b\) itself, and minimises the squared error \(L = \tfrac{1}{2}(y - z)^2\) by gradient descent. Since \(\partial L / \partial \mathbf{w} = -(y - z)\,\mathbf{x}\), the update is

<div class="eq">\[ \mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - z)\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - z), \]
<span class="where">the <strong>delta rule</strong> (Widrow–Hoff rule, LMS rule), named after the error \(\delta = y - z\).</span></div>

Three things change relative to the perceptron rule: the update applies to <em>every</em> example, scaled by how wrong the linear output is; the loss is smooth and convex, so gradient descent converges to its unique minimum whether or not the data are separable; and the step size \(\eta\) now matters — too large diverges, too small crawls (Section 9.1). Replace the squared error by the cross-entropy and the linear output by a sigmoid and the Adaline becomes the logistic regression of Unit 2; Section 8 generalises its "error times input" to hidden units whose error is not directly observable. The chunk applies the Adaline to the same two features with three learning rates.



In [ ]:
# Adaline (delta rule): full-batch gradient descent on the squared error, three learning rates.
adaline_fit <- function(X, y, eta, epochs = 60) {
  w <- rep(0, ncol(X)); b <- 0; losses <- numeric(epochs)
  for (e in 1:epochs) {
    z <- X %*% w + b; err <- y - z
    w <- w + eta * t(X) %*% err / length(y); b <- b + eta * mean(err)
    losses[e] <- 0.5 * mean((y - (X %*% w + b))^2)
  }
  list(w = w, b = b, losses = losses)
}
rho <- cor(Z2)[1, 2]
cat(sprintf("correlation of the two features = %.3f -> divergence for eta > 2/(1+rho) = %.2f\n", rho, 2 / (1 + rho)))
plot(NA, xlim = c(1, 60), ylim = c(0.1, 1e3), log = "y", xlab = "epoch", ylab = "squared-error loss (log scale)",
     main = "Adaline: crawling, converging and diverging")
for (cfg in list(c(0.01, 1), c(0.3, 2), c(1.25, 3))) {
  eta <- cfg[1]; fit <- adaline_fit(Z2, y_pm, eta)
  acc <- mean(ifelse(Z2 %*% fit$w + fit$b >= 0, 1, -1) == y_pm)
  cat(sprintf("eta = %-5g: loss after 60 epochs = %10.3g, training accuracy = %.3f\n", eta, fit$losses[60], acc))
  lines(1:60, fit$losses, col = c(GREY, BLUE, ORANGE)[cfg[2]], lwd = 1.4)
}
legend("topright", c("eta = 0.01", "eta = 0.3", "eta = 1.25"), col = c(GREY, BLUE, ORANGE), lwd = 1.4, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 9.</b> Loss per epoch of the Adaline for three learning rates. With \(\eta = 0.01\) the loss decreases but has not converged after 60 epochs; with \(\eta = 0.3\) it reaches its minimum within about twenty; with \(\eta = 1.25\), just above the stability threshold, each step overshoots the minimum by more than it gains and the loss <em>grows</em> — divergence. For a convex quadratic loss the threshold is exact: full-batch gradient descent diverges when \(\eta\) exceeds \(2/\lambda_{\max}\), where \(\lambda_{\max}\) is the largest eigenvalue of the feature covariance matrix; for two standardised features with correlation \(\rho\), \(\lambda_{\max} = 1 + \rho\), which gives the threshold printed above the figure. The Adaline reaches an accuracy close to the perceptron's, but by a procedure that stops.</p>

<div class="box bridge"><h4>Bridge to Unit 2</h4><p>The Adaline is exactly the linear regression of Unit 2, fitted by gradient descent rather than by the normal equations, with the class coded as \(\pm 1\). The delta rule, Unit 2's gradient step for logistic regression and the backpropagation of the next section are the same instruction — <em>move each weight in proportion to the error times the input it multiplies</em> — applied to models of increasing depth.</p></div>

## 8. The multilayer perceptron

### 8.1 Architecture and forward propagation

A <strong>multilayer perceptron</strong> is a feed-forward network with at least one hidden layer, every layer fully connected to the next. Despite the name, its units are not perceptrons in Rosenblatt's sense: they use differentiable activations (sigmoid, tanh, ReLU) rather than the step, precisely so that the network can be trained by gradient descent. The computation of layer \(l\), for all \(n\) training cases at once, is one matrix product followed by an element-wise activation:

<div class="eq">\[ \mathbf{Z}^{(l)} = \mathbf{A}^{(l-1)} \mathbf{W}^{(l)} + \mathbf{b}^{(l)}, \qquad \mathbf{A}^{(l)} = \varphi^{(l)}\!\left(\mathbf{Z}^{(l)}\right), \qquad \mathbf{A}^{(0)} = \mathbf{X}, \]
<span class="where">where \(\mathbf{X}\) is the \(n \times n_0\) data matrix, \(\mathbf{W}^{(l)}\) is \(n_{l-1} \times n_l\), \(\mathbf{b}^{(l)}\) is a row of \(n_l\) biases added to every row (recycled across rows), \(\mathbf{Z}^{(l)}\) and \(\mathbf{A}^{(l)}\) are \(n \times n_l\). The output \(\hat{\mathbf{Y}} = \mathbf{A}^{(L)}\) is \(n \times n_L\). This is <strong>forward propagation</strong>.</span></div>

For the 9-16-1 network on the breast-cancer data: \(\mathbf{X}\) is \(683 \times 9\), \(\mathbf{W}^{(1)}\) is \(9 \times 16\), \(\mathbf{A}^{(1)}\) is \(683 \times 16\) (the sixteen learned features of every case), \(\mathbf{W}^{(2)}\) is \(16 \times 1\) and \(\hat{\mathbf{y}}\) is a column of 683 probabilities.

### 8.2 Loss functions

Training needs a number that says how wrong the network is on the training sample, to be minimised. For <strong>regression</strong> with a linear output unit the natural choice is the <strong>mean squared error</strong>, \(L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2\), as in Unit 2. For <strong>binary classification</strong> with a sigmoid output interpreted as \(\hat{y} = P(y = 1 \mid \mathbf{x})\), the loss is the <strong>binary cross-entropy</strong> (negative log-likelihood of the Bernoulli model),

<div class="eq">\[ L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \ln \hat{y}_i + (1 - y_i) \ln (1 - \hat{y}_i) \right], \]</div>

and for \(K\) classes with a softmax output and one-hot targets \(y_{ik}\), the <strong>categorical cross-entropy</strong> \(L = -\frac{1}{n}\sum_i \sum_k y_{ik} \ln \hat{y}_{ik}\). Cross-entropy is preferred over squared error for classification for a reason that becomes visible in the derivation below: with a sigmoid output and cross-entropy, the error signal at the output is simply \(\hat{y} - y\), whereas with squared error it is \((\hat{y} - y)\,\sigma'(z)\), which vanishes when the output unit is saturated — a confidently wrong network would then learn almost nothing from its mistake. The same simplification holds for softmax with categorical cross-entropy: the derivative of the loss with respect to the logits is \(\hat{\mathbf{y}} - \mathbf{y}\). The hand trace of Section 8.4 uses the squared error so that every factor is visible; the general implementation of Section 8.6 uses cross-entropy.

### 8.3 Backpropagation: the chain rule, organised

The minimisation is by gradient descent, \(\theta \leftarrow \theta - \eta\, \partial L / \partial \theta\) for every weight and bias, so the whole problem is to compute the partial derivatives. For a network with hidden layers that is a bookkeeping problem, and <strong>backpropagation</strong> is the bookkeeping: an efficient organisation of the chain rule, discovered several times and brought to the field's attention by Rumelhart, Hinton and Williams in 1986.<sup class="c"><a href="#r4">[4]</a></sup> Its idea is to define, for every unit, the derivative of the loss with respect to that unit's <em>pre-activation</em>, \(\delta = \partial L / \partial z\), and to notice that the \(\delta\) of a layer can be computed from the \(\delta\) of the next layer — so the derivatives are computed backwards, from the output to the input, in one pass that costs about as much as the forward pass.

Take a 2-2-1 network with sigmoid activations everywhere and the squared-error loss on one example, \(L = \tfrac{1}{2}(a^{(2)} - y)^2\). Forward: \(z_j^{(1)} = \sum_i W^{(1)}_{ij} x_i + b^{(1)}_j\), \(a_j^{(1)} = \sigma(z_j^{(1)})\) for \(j = 1, 2\); \(z^{(2)} = \sum_j W^{(2)}_j a_j^{(1)} + b^{(2)}\), \(a^{(2)} = \sigma(z^{(2)})\). Backward, by the chain rule:

**Output unit.** \(\delta^{(2)} = \dfrac{\partial L}{\partial z^{(2)}} = (a^{(2)} - y)\; a^{(2)} (1 - a^{(2)})\). Then \(\dfrac{\partial L}{\partial W^{(2)}_j} = \delta^{(2)} a_j^{(1)}\) and \(\dfrac{\partial L}{\partial b^{(2)}} = \delta^{(2)}\) — "error times input", the delta rule.

**Hidden units.** Hidden unit \(j\) affects \(L\) only through \(z^{(2)}\), so \(\delta^{(1)}_j = \delta^{(2)}\, W^{(2)}_j\; a_j^{(1)}(1 - a_j^{(1)})\): the output error is sent <em>back along the weight</em> that connects the two units and scaled by the derivative of the hidden activation. Then \(\dfrac{\partial L}{\partial W^{(1)}_{ij}} = \delta^{(1)}_j x_i\) and \(\dfrac{\partial L}{\partial b^{(1)}_j} = \delta^{(1)}_j\).

The pattern generalises to any depth. In matrix form, for a mini-batch, with \(\odot\) the element-wise product:

<div class="eq">\[ \boldsymbol{\Delta}^{(L)} = \frac{\partial L}{\partial \mathbf{Z}^{(L)}}, \quad \boldsymbol{\Delta}^{(l)} = \left(\boldsymbol{\Delta}^{(l+1)} \mathbf{W}^{(l+1)\top}\right) \odot \varphi'\!\left(\mathbf{Z}^{(l)}\right), \quad \frac{\partial L}{\partial \mathbf{W}^{(l)}} = \mathbf{A}^{(l-1)\top} \boldsymbol{\Delta}^{(l)}, \quad \frac{\partial L}{\partial \mathbf{b}^{(l)}} = \mathbf{1}^\top \boldsymbol{\Delta}^{(l)}. \]
<span class="where">For a sigmoid output with cross-entropy, or a softmax output with categorical cross-entropy, \(\boldsymbol{\Delta}^{(L)} = (\hat{\mathbf{Y}} - \mathbf{Y})/n\). These four lines are the whole of the <code>backward</code> function in Section 8.6.</span></div>

### 8.4 Worked example by hand: one training step of the 2-2-1 network

Take the XOR example \(\mathbf{x} = (1, 1)\) with target \(y = 0\), learning rate \(\eta = 0.5\), and the weights

<div class="eq">\[ \mathbf{W}^{(1)} = \begin{pmatrix} 0.5 & -0.5 \\ 0.3 & 0.8 \end{pmatrix}, \quad \mathbf{b}^{(1)} = (0.1, -0.2), \quad \mathbf{W}^{(2)} = (0.6, -0.4), \quad b^{(2)} = 0.2, \]</div>

with the convention that <em>row \(j\) of \(\mathbf{W}^{(1)}\) holds the weights into hidden unit \(j\)</em>, so that \(\mathbf{z}^{(1)} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\). (The matrix form of Section 8.1 uses the transpose, \(\mathbf{X}\mathbf{W}\), because there the cases are rows; textbooks differ, and the only rule is to use one convention consistently.)

**Forward pass.** \(z^{(1)} = (0.1,\ 0.9)\), \(a^{(1)} = (\sigma(0.1), \sigma(0.9)) = (0.5250,\ 0.7109)\); \(z^{(2)} = 0.6 \times 0.5250 - 0.4 \times 0.7109 + 0.2 = 0.2306\), \(a^{(2)} = \sigma(0.2306) = 0.5574\); \(L = \tfrac{1}{2}(0.5574)^2 = 0.1553\).

**Backward pass.** \(\delta^{(2)} = (a^{(2)} - y)\,a^{(2)}(1 - a^{(2)}) = 0.5574 \times 0.5574 \times 0.4426 = 0.1375\). \(\partial L/\partial W^{(2)} = \delta^{(2)} a^{(1)} = (0.0722,\ 0.0978)\); \(\partial L/\partial b^{(2)} = 0.1375\). \(\delta^{(1)}_1 = 0.1375 \times 0.6 \times 0.5250 \times 0.4750 = 0.0206\); \(\delta^{(1)}_2 = 0.1375 \times (-0.4) \times 0.7109 \times 0.2891 = -0.0113\). \(\partial L/\partial W^{(1)}_{ji} = \delta^{(1)}_j x_i\): both inputs are 1, so row 1 of the gradient is \((0.0206, 0.0206)\) and row 2 is \((-0.0113, -0.0113)\); \(\partial L/\partial b^{(1)} = (0.0206, -0.0113)\).

**Update** (\(\theta \leftarrow \theta - 0.5\, \partial L/\partial\theta\)). \(W^{(2)} \leftarrow (0.5639,\ -0.4489)\), \(b^{(2)} \leftarrow 0.1312\); \(W^{(1)} \leftarrow \begin{pmatrix} 0.4897 & -0.5103 \\ 0.3057 & 0.8057 \end{pmatrix}\), \(b^{(1)} \leftarrow (0.0897,\ -0.1943)\).

Note the direction of every change: the output was 0.5574 for a target of 0, so \(W^{(2)}_1\), which multiplies a positive hidden activation, decreased, and \(W^{(2)}_2\), which was already negative, became more negative — both push \(z^{(2)}\) down. The hidden-layer changes are an order of magnitude smaller than the output-layer ones: the error has been multiplied by a weight of magnitude \(\leq 0.6\) and by a sigmoid derivative \(\leq 0.25\) on its way back. One more layer would shrink it again; this is the vanishing gradient of Section 9.6 in miniature. The chunk reproduces every number and then checks the gradient against a finite difference.



In [ ]:
# Backpropagation by hand, in code: forward pass, deltas, gradients and the update for the 2-2-1 example.
x_ex <- c(1, 1); y_ex <- 0; eta <- 0.5
W1 <- matrix(c(0.5, -0.5, 0.3, 0.8), nrow = 2, byrow = TRUE); b1 <- c(0.1, -0.2)   # row j = weights INTO hidden unit j
W2 <- c(0.6, -0.4); b2 <- 0.2
z1 <- as.vector(W1 %*% x_ex + b1); a1 <- sigmoid(z1)                                # hidden layer
z2 <- sum(W2 * a1) + b2;           a2 <- sigmoid(z2)                                # output unit
loss <- 0.5 * (a2 - y_ex)^2
cat(sprintf("forward : z1 = (%.4f, %.4f), a1 = (%.4f, %.4f), z2 = %.4f, a2 = %.4f, loss = %.4f\n",
            z1[1], z1[2], a1[1], a1[2], z2, a2, loss))
d2 <- (a2 - y_ex) * a2 * (1 - a2); gW2 <- d2 * a1; gb2 <- d2                        # delta at the output
d1 <- d2 * W2 * a1 * (1 - a1); gW1 <- outer(d1, x_ex); gb1 <- d1                    # deltas at the hidden units
cat(sprintf("backward: delta2 = %.4f, dL/dW2 = (%.4f, %.4f)\n", d2, gW2[1], gW2[2]))
cat(sprintf("          delta1 = (%.4f, %.4f)\n", d1[1], d1[2]))
cat(sprintf("update  : W2 -> (%.4f, %.4f), b2 -> %.4f\n", (W2 - eta * gW2)[1], (W2 - eta * gW2)[2], b2 - eta * gb2))
cat("          W1 ->\n"); print(round(W1 - eta * gW1, 4))
loss_of <- function(W1_) 0.5 * (sigmoid(sum(W2 * sigmoid(as.vector(W1_ %*% x_ex + b1))) + b2) - y_ex)^2
h <- 1e-6; Wp <- W1; Wm <- W1; Wp[1, 1] <- Wp[1, 1] + h; Wm[1, 1] <- Wm[1, 1] - h
cat(sprintf("check   : dL/dW1[1,1] backprop = %.7f, finite difference = %.7f\n",
            gW1[1, 1], (loss_of(Wp) - loss_of(Wm)) / (2 * h)))



Every number of the hand trace is reproduced to four decimals, and the finite-difference derivative — the loss recomputed with one weight nudged up and down — agrees with the backpropagated one to seven decimals. The finite-difference check is too slow for training (two forward passes per parameter) but is the standard way to verify an implementation, and Section 8.7 applies it to the general class.

### 8.5 Training XOR to convergence

Repeating the step of Section 8.4 on all four XOR patterns, thousands of times, is training. The chunk below does it with full-batch gradient descent on the squared error and the same 2-2-1 architecture, starting from random weights, and then shows the function the network has learned over the whole unit square.



In [ ]:
# Training the 2-2-1 sigmoid network on XOR by full-batch gradient descent (squared-error loss, as in the hand trace).
set.seed(0)
X_xor01 <- Xb; y_xor01 <- c(0, 1, 1, 0)
W1 <- matrix(rnorm(4), 2, 2); b1 <- c(0, 0); W2 <- matrix(rnorm(2), 2, 1); b2 <- 0; losses <- numeric(10000)
for (epoch in 1:10000) {
  A1 <- sigmoid(X_xor01 %*% W1 + matrix(b1, 4, 2, byrow = TRUE)); A2 <- sigmoid(A1 %*% W2 + b2)   # forward
  D2 <- (A2 - y_xor01) * A2 * (1 - A2) / 4                                                         # output deltas
  D1 <- (D2 %*% t(W2)) * A1 * (1 - A1)                                                             # hidden deltas
  W2 <- W2 - t(A1) %*% D2; b2 <- b2 - sum(D2); W1 <- W1 - t(X_xor01) %*% D1; b1 <- b1 - colSums(D1)
  losses[epoch] <- 0.5 * mean((as.vector(A2) - y_xor01)^2)
}
out <- as.vector(sigmoid(sigmoid(X_xor01 %*% W1 + matrix(b1, 4, 2, byrow = TRUE)) %*% W2 + b2))
cat("loss at epochs 1, 1000, 5000, 10000:", round(losses[c(1, 1000, 5000, 10000)], 4), "\n")
cat("network output for (0,0),(0,1),(1,0),(1,1):", round(out, 3), "-> rounded:", as.integer(out >= 0.5), "\n")
gx <- seq(-0.5, 1.5, length.out = 120); gr <- as.matrix(expand.grid(x1 = gx, x2 = gx))
go <- sigmoid(sigmoid(gr %*% W1 + matrix(b1, nrow(gr), 2, byrow = TRUE)) %*% W2 + b2)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(losses, type = "l", col = BLUE, log = "x", xlab = "epoch (log scale)", ylab = "squared-error loss", main = "Training loss on XOR")
image(gx, gx, matrix(go, 120, 120), col = hcl.colors(20, "RdYlBu", rev = TRUE), xlab = "x1", ylab = "x2", main = "Learned function")
contour(gx, gx, matrix(go, 120, 120), levels = 0.5, add = TRUE, lwd = 1.4)
points(Xb[y_xor01 == 0, , drop = FALSE], pch = 21, bg = "white", cex = 1.6); points(Xb[y_xor01 == 1, , drop = FALSE], pch = 21, bg = "black", cex = 1.6)



<p class="caption"><b>Figure 10.</b> Left: the squared-error loss during 10,000 epochs of gradient descent on XOR. The loss sits on a plateau near 0.125 for the first few thousand epochs — the output is about 0.5 for every pattern — and then drops sharply once the hidden units have differentiated. Right: the learned function over the unit square. The output unit assigns low output to the band that contains \((0,0)\) and \((1,1)\) (white dots) and high output where \((0,1)\) and \((1,0)\) lie (black dots). The decision boundary (output = 0.5) consists of two lines — exactly the geometry Section 6.3 proved a single unit cannot produce.</p>

The printed weights show how the network solved the problem: each hidden unit implements a linearly separable sub-function and the output unit combines them. Which decomposition gradient descent finds depends on the starting weights, a first instance of the non-convexity of Section 4.2. Minsky and Papert knew in 1969 that this architecture <em>represents</em> XOR; what they doubted was that any procedure could <em>learn</em> the hidden-layer weights from input–output pairs alone, since nobody tells the hidden units what they should compute.<sup class="c"><a href="#r3">[3]</a></sup> Backpropagation answers the doubt: the hidden units receive their error signal from the output through the chain rule.

<div class="box key"><h4>Exercise 3</h4><p>Change the seed in <code>set.seed(0)</code> to 1, 2, 3, 4 and re-run. Does the network always converge in 10,000 epochs? Compare the final losses and the learned functions. Then replace the squared-error gradient <code>(A2 - y) * A2 * (1 - A2)</code> by the cross-entropy gradient <code>(A2 - y)</code> and observe how the plateau at the beginning of training changes. Section 8.2 explains why.</p></div>

### 8.6 A general MLP in base R

The XOR code hard-wires two layers. The functions below implement the matrix equations of Sections 8.1 and 8.3 for any vector of layer sizes, with a choice of hidden activation (sigmoid or ReLU), Glorot or Gaussian initialisation (Section 9.3), an optional L2 penalty (Section 9.5), and a sigmoid output trained with the binary cross-entropy. `nnet` does the same with a better optimiser (BFGS) and is used from Section 9.7 onwards; these functions are the ones to read line by line against the equations. They separate the <em>model</em> — how to compute the output (`mlp_forward`) and the gradient (`mlp_backward`) — from the <em>training loop</em> `mlp_train`, which implements <strong>mini-batch stochastic gradient descent</strong>: each epoch shuffles the training cases, cuts them into batches, takes one gradient step per batch, and records the training loss (and the validation loss, if a validation set is given) after every epoch — the <em>loss curves</em> that are the main diagnostic instrument of Section 9.



In [ ]:
# A general multilayer perceptron for binary classification: init, forward, backward (base R, numpy-style matrices).
mlp_init <- function(sizes, act = "relu", init = "glorot", l2 = 0, seed = 0) {
  set.seed(seed); W <- list(); b <- list()
  for (l in seq_len(length(sizes) - 1)) {
    n_in <- sizes[l]; n_out <- sizes[l + 1]
    if (identical(init, "glorot")) { lim <- sqrt(6 / (n_in + n_out)); W[[l]] <- matrix(runif(n_in * n_out, -lim, lim), n_in, n_out) }
    else W[[l]] <- matrix(rnorm(n_in * n_out, 0, init), n_in, n_out)
    b[[l]] <- rep(0, n_out)
  }
  list(W = W, b = b, act = act, l2 = l2)
}
phi  <- function(net, z) if (net$act == "relu") pmax(z, 0) else sigmoid(z)
dphi <- function(net, z, a) if (net$act == "relu") (z > 0) * 1 else a * (1 - a)
mlp_forward <- function(net, X) {                             # returns Z, A (list) and the output vector
  Z <- list(); A <- list(X)
  for (l in seq_along(net$W)) {
    z <- A[[l]] %*% net$W[[l]] + matrix(net$b[[l]], nrow(X), length(net$b[[l]]), byrow = TRUE)
    Z[[l]] <- z; A[[l + 1]] <- if (l == length(net$W)) sigmoid(z) else phi(net, z)
  }
  list(Z = Z, A = A, out = as.vector(A[[length(A)]]))
}
mlp_loss <- function(net, X, y) {
  p <- pmin(pmax(mlp_forward(net, X)$out, 1e-12), 1 - 1e-12)
  pen <- if (net$l2) 0.5 * net$l2 * sum(sapply(net$W, function(W) sum(W^2))) else 0
  -mean(y * log(p) + (1 - y) * log(1 - p)) + pen
}
mlp_backward <- function(net, X, y) {                         # returns dL/dW and dL/db for every layer
  n <- length(y); fp <- mlp_forward(net, X); A <- fp$A; Z <- fp$Z
  delta <- (A[[length(A)]] - y) / n                           # Delta(L) = (Yhat - Y)/n for sigmoid + cross-entropy
  gW <- vector("list", length(net$W)); gb <- vector("list", length(net$W))
  for (l in length(net$W):1) {
    gW[[l]] <- t(A[[l]]) %*% delta + net$l2 * net$W[[l]]; gb[[l]] <- colSums(delta)
    if (l > 1) delta <- (delta %*% t(net$W[[l]])) * dphi(net, Z[[l - 1]], A[[l]])
  }
  list(gW = gW, gb = gb)
}

# --- The training loop: mini-batch stochastic gradient descent, loss recorded after every epoch. ---
mlp_train <- function(net, X, y, lr = 0.1, epochs = 100, batch = 32, Xval = NULL, yval = NULL, seed = 0) {
  set.seed(seed); hist <- list(train = numeric(0), val = numeric(0))
  for (ep in 1:epochs) {
    idx <- sample(length(y)); folds <- split(idx, ceiling(seq_along(idx) / batch))
    for (b in folds) {
      g <- mlp_backward(net, X[b, , drop = FALSE], y[b])
      for (l in seq_along(net$W)) { net$W[[l]] <- net$W[[l]] - lr * g$gW[[l]]; net$b[[l]] <- net$b[[l]] - lr * g$gb[[l]] }
    }
    hist$train <- c(hist$train, mlp_loss(net, X, y))
    if (!is.null(Xval)) hist$val <- c(hist$val, mlp_loss(net, Xval, yval))
    if (!is.finite(tail(hist$train, 1))) break
  }
  list(net = net, hist = hist)
}
# Breast cancer, all 9 features: 80% train / 20% test, standardised with training-set statistics only.
set.seed(0); feat9 <- as.matrix(bc[, 1:9]); n <- nrow(feat9); tr <- sample(n, round(0.8 * n))
mu <- colMeans(feat9[tr, ]); sdv <- apply(feat9[tr, ], 2, sd)
Ztr <- scale(feat9[tr, ], mu, sdv); Zte <- scale(feat9[-tr, ], mu, sdv); ytr <- y_mal[tr]; yte <- y_mal[-tr]
cat(sprintf("training cases %d (malignant %d), test cases %d (malignant %d)\n", length(ytr), sum(ytr), length(yte), sum(yte)))
plot(NA, xlim = c(1, 100), ylim = c(0.03, 0.8), log = "y", xlab = "epoch", ylab = "cross-entropy loss", main = "Loss curves of the base-R MLP (breast cancer)")
for (cfg in list(c("sigmoid", ORANGE), c("relu", BLUE))) {
  res <- mlp_train(mlp_init(c(9, 16, 1), act = cfg[1]), Ztr, ytr, lr = 0.1, epochs = 100, Xval = Zte, yval = yte)
  lines(res$hist$train, col = cfg[2], lwd = 1.4); lines(res$hist$val, col = cfg[2], lwd = 1.4, lty = 2)
  acc <- mean((mlp_forward(res$net, Zte)$out >= 0.5) == yte)
  cat(sprintf("9-16-1 %-7s: final train loss %.4f, test loss %.4f, test accuracy %.3f\n",
              cfg[1], tail(res$hist$train, 1), tail(res$hist$val, 1), acc))
}
legend("topright", c("sigmoid train", "relu train", "(dashed = test)"), col = c(ORANGE, BLUE, GREY), lwd = 1.4, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 11.</b> Training (solid) and test (dashed) cross-entropy of the 9-16-1 network with sigmoid and with ReLU hidden units, over 100 epochs of mini-batch SGD with learning rate 0.1. Both networks reach a test accuracy around 0.95 (sigmoid) to 0.96 (ReLU), close to the level of the best methods of Units 3–4 on these data (Section 10 tunes them properly). The ReLU network descends faster in the first epochs, as its non-saturating derivative predicts. The test loss flattens while the training loss keeps falling: the network is beginning to fit the training sample better than new cases, the overfitting that Section 9.2 addresses with early stopping.</p>

### 8.7 Gradient checking

Backpropagation code is easy to get subtly wrong — a transposed matrix, a missing derivative, an off-by-one layer index — and a wrong gradient often still decreases the loss, slowly, so the bug goes unnoticed. The standard test compares the analytical gradient with the centred finite difference \(\left[L(\theta + \epsilon) - L(\theta - \epsilon)\right] / 2\epsilon\) for a sample of parameters, and reports the relative error \(|g_{\text{bp}} - g_{\text{fd}}| / (|g_{\text{bp}}| + |g_{\text{fd}}|)\). A correct implementation gives relative errors below about \(10^{-7}\) in double precision; a bug gives errors of order 1.



In [ ]:
# Gradient checking: backpropagation versus centred finite differences, for random weights of each layer.
set.seed(1); net_chk <- mlp_init(c(9, 8, 4, 1), act = "sigmoid", seed = 1)
Xs <- Ztr[1:25, ]; ys <- ytr[1:25]; g <- mlp_backward(net_chk, Xs, ys); eps <- 1e-5; rows <- list()
for (l in seq_along(net_chk$W)) for (rep in 1:4) {
  i <- sample(nrow(net_chk$W[[l]]), 1); j <- sample(ncol(net_chk$W[[l]]), 1); orig <- net_chk$W[[l]][i, j]
  net_chk$W[[l]][i, j] <- orig + eps; Lp <- mlp_loss(net_chk, Xs, ys)
  net_chk$W[[l]][i, j] <- orig - eps; Lm <- mlp_loss(net_chk, Xs, ys); net_chk$W[[l]][i, j] <- orig
  gfd <- (Lp - Lm) / (2 * eps); gbp <- g$gW[[l]][i, j]
  rows[[length(rows) + 1]] <- data.frame(layer = l, weight = sprintf("W[%d,%d]", i, j),
      backprop = gbp, finite_diff = gfd, rel_error = abs(gbp - gfd) / (abs(gbp) + abs(gfd) + 1e-15))
}
chk <- do.call(rbind, rows); print(chk, row.names = FALSE, digits = 3)
cat(sprintf("\nlargest relative error: %.2e\n", max(chk$rel_error)))



All relative errors are small — most of order \(10^{-8}\) or below — so the implementation of the four backpropagation equations is correct for a three-layer network with 9-dimensional input, including the layers that the XOR example never exercised.

<div class="box key"><h4>Key idea</h4><p>Backpropagation is not a learning rule but a method for computing derivatives: the chain rule applied layer by layer from the output backwards, reusing the quantity \(\delta^{(l)} = \partial L / \partial z^{(l)}\) so that the cost of the whole gradient is about twice the cost of one forward pass. Gradient descent then uses the derivatives. Every modern framework automates exactly the computation of Section 8.6 — for any architecture, by recording the forward operations and differentiating them in reverse (automatic differentiation).</p></div>

## 9. Training in practice

Backpropagation gives the gradient; everything else about training a network is a set of choices that the theory does not make for you. This section goes through them in the order in which they usually cause trouble, with one experiment each, using the base-R functions on the breast-cancer data and then `nnet`.

### 9.1 The learning rate

The learning rate \(\eta\) is the most important hyperparameter of plain gradient descent. Too small, and training takes thousands of epochs to make progress; too large, and each step overshoots the minimum, the loss oscillates and can even diverge to infinity — the behaviour seen for the Adaline in Figure 9, now in a non-convex problem where no formula gives the threshold. The usual practice is to try values spaced by factors of 10, pick the largest that decreases the loss smoothly, and reduce it during training. Adaptive methods such as Adam, which keep a running estimate of the gradient's mean and variance and scale the step per parameter,<sup class="c"><a href="#r28">[28]</a></sup> are the default in the deep-learning frameworks of Unit 8. R's `nnet` sidesteps the choice by using a quasi-Newton optimiser (BFGS) that sets its own step length, and `neuralnet` uses resilient backpropagation (rprop); the from-scratch loop below uses a fixed \(\eta\) so that the phenomenon is visible.



In [ ]:
# Three learning rates on the 9-16-1 ReLU network: too small (crawls), adequate, too large (diverges).
plot(NA, xlim = c(1, 60), ylim = c(0.02, 50), log = "y", xlab = "epoch", ylab = "training cross-entropy (log scale)",
     main = "Learning rate: crawling, converging, diverging")
for (cfg in list(c(0.001, 1), c(0.1, 2), c(10, 3))) {
  lr <- cfg[1]; res <- mlp_train(mlp_init(c(9, 16, 1), act = "relu"), Ztr, ytr, lr = lr, epochs = 60)
  fin <- tail(res$hist$train, 1)
  status <- if (!is.finite(fin)) "diverged (loss became inf/nan)" else sprintf("final loss %.4f", fin)
  cat(sprintf("lr = %-6g: %2d epochs run, loss after epoch 1 = %.3f, %s\n", lr, length(res$hist$train), res$hist$train[1], status))
  lines(seq_along(res$hist$train), res$hist$train, type = "o", pch = 19, cex = 0.5, col = c(GREY, BLUE, ORANGE)[cfg[2]], lwd = 1.3)
}
legend("topright", c("eta = 0.001", "eta = 0.1", "eta = 10"), col = c(GREY, BLUE, ORANGE), lwd = 1.3, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 12.</b> Training loss for three learning rates. At \(\eta = 0.001\) the loss has barely moved from its initialisation after 60 epochs (it crawls); at \(\eta = 0.1\) it falls smoothly to a small value; at \(\eta = 10\) the first step throws the loss up above 1 — the steps overshoot the minimum — and although the run partially recovers it stays erratic and well above the \(\eta = 0.1\) curve. A still larger rate makes the weights overflow and the loss become undefined, at which point the training loop stops the run (the <code>is.finite</code> guard). This is the non-convex counterpart of the Adaline's exact threshold in Figure 9.</p>

### 9.2 Epochs and early stopping

How long to train is the second choice, and it is best made by watching a <strong>validation set</strong> — a part of the training data held out from the gradient steps — rather than fixed in advance. Typically the validation loss decreases with the training loss at first, then flattens and begins to rise while the training loss continues to fall: from that point on the network is fitting the particular training cases rather than the pattern they share. <strong>Early stopping</strong> is the rule "stop when the validation loss has not improved for \(k\) epochs (the <em>patience</em>) and keep the weights from the best epoch". It is the cheapest and most used form of regularisation for neural networks. `nnet` has no early-stopping option — its BFGS optimiser runs to `maxit` or its tolerance — so the experiment below uses the from-scratch loop, which records the validation loss every epoch; it exaggerates the phenomenon by training a wide network (9-256-1) on only 80 cases.



In [ ]:
# Overfitting and early stopping: a wide network trained on a small sample; validation loss turns upward.
set.seed(0); small <- sample(length(ytr), 80)
res <- mlp_train(mlp_init(c(9, 256, 1), act = "relu"), Ztr[small, ], ytr[small], lr = 0.1, epochs = 1500, batch = 16, Xval = Zte, yval = yte)
val <- res$hist$val; best <- which.min(val)
cat(sprintf("%d parameters trained on 80 cases\n", n_parameters(c(9, 256, 1))))
cat(sprintf("validation loss: minimum %.4f at epoch %d, final %.4f | training loss at end %.5f\n",
            min(val), best, tail(val, 1), tail(res$hist$train, 1)))
plot(res$hist$train, type = "l", col = BLUE, lwd = 1.3, log = "xy", xlab = "epoch (log)", ylab = "cross-entropy (log)",
     main = "Overfitting: the two loss curves separate", ylim = range(c(res$hist$train, val)))
lines(val, col = ORANGE, lwd = 1.3); abline(v = best, col = GREY, lty = 2)
legend("bottomleft", c("training (80 cases)", "validation (held out)", sprintf("early-stop epoch %d", best)),
       col = c(BLUE, ORANGE, GREY), lwd = c(1.3, 1.3, 1), lty = c(1, 1, 2), bty = "n", cex = 0.8)



<p class="caption"><b>Figure 13.</b> Training and validation loss of a 9-256-1 network trained on 80 cases for 1,500 epochs. The training loss decreases towards zero — the network can memorise 80 cases with thousands of parameters without difficulty. The validation loss reaches its minimum early and then rises: every further epoch makes the network <em>worse</em> on new patients while making it better on the training ones. Early stopping would have kept the weights from the marked epoch. The log–log axes show that the separation begins early and continues for the whole run.</p>

### 9.3 Weight initialisation

Gradient descent must start somewhere, and the start matters more for networks than for convex models. Starting all weights at zero is fatal: every hidden unit then receives the same input and the same gradient, so all units of a layer remain identical for ever — the <em>symmetry</em> problem. Random weights break the symmetry, but their <em>scale</em> matters too: if too large, sigmoid and tanh units saturate at once and their gradients vanish; if too small, the signal shrinks layer by layer. Glorot and Bengio analysed the forward and backward variance across layers and proposed weights drawn uniformly from \(\pm\sqrt{6 / (n_{\text{in}} + n_{\text{out}})}\), which keeps both roughly constant through a stack of tanh layers;<sup class="c"><a href="#r25">[25]</a></sup> this <strong>Glorot</strong> (Xavier) initialisation is the one the `mlp_init` function uses by default. For ReLU units, which discard half of their inputs, the variance should be doubled (He initialisation). Section 9.6 shows what the choice does to the gradients of a deep network.

### 9.4 Input standardisation

Networks are trained by gradient steps of a single size in every direction of parameter space, and a feature whose scale is a thousand times another's produces gradients a thousand times larger for its weights, so no single step suits both. Standardising every feature to mean 0 and standard deviation 1 is therefore not optional for networks, as it is for trees: it is part of the model. The breast-cancer features all share the 1–10 ordinal scale, so to make the point vividly the chunk multiplies one feature by 1000 before fitting, and compares the cross-validated accuracy with and without standardisation. The scaler must be fitted on the training folds only — fitting it on all the data would leak the test cases' statistics into training.



In [ ]:
# Input standardisation: the same nnet with raw (one feature blown up x1000) and standardised features, 5-fold CV.
set.seed(0)
Xraw <- as.matrix(bc[, 1:9]); Xraw[, 1] <- Xraw[, 1] * 1000           # exaggerate the scale mismatch
folds <- sample(rep(1:5, length.out = nrow(Xraw)))
cv_nnet_acc <- function(X, standardise) {
  acc <- sapply(1:5, function(k) {
    trk <- folds != k; tek <- !trk; Xt <- X[trk, ]; Xv <- X[tek, ]
    if (standardise) { m <- colMeans(Xt); s <- apply(Xt, 2, sd); Xt <- scale(Xt, m, s); Xv <- scale(Xv, m, s) }
    fit <- nnet(Xt, y_mal[trk], size = 16, entropy = TRUE, maxit = 200, trace = FALSE, MaxNWts = 5000)
    mean((predict(fit, Xv) >= 0.5) == y_mal[tek])
  })
  acc
}
raw <- cv_nnet_acc(Xraw, FALSE); std <- cv_nnet_acc(Xraw, TRUE)
cat(sprintf("raw features (1 blown up x1000): CV accuracy %.3f +/- %.3f\n", mean(raw), sd(raw)))
cat(sprintf("standardised features          : CV accuracy %.3f +/- %.3f\n", mean(std), sd(std)))



On the raw features the optimiser struggles: the gradients of the weights attached to the inflated feature dwarf the others, and the network either fails to learn or predicts the majority class, so its cross-validated accuracy is low and unstable. The same architecture and folds on standardised features give an accuracy near 0.96. Everything from here on standardises inside the cross-validation, fitting the scaling on the training folds only.

### 9.5 Regularisation: weight decay and dropout

Early stopping limits the <em>time</em> the network has to fit noise; regularisation limits its <em>capacity</em> to do so. The <strong>L2 penalty</strong> (weight decay) adds \(\frac{\alpha}{2}\sum \theta^2\) to the loss, so that each gradient step also shrinks every weight towards zero; large weights, which produce sharp, confident functions, are discouraged. It is the `decay` argument of `nnet` and the `l2` argument of the base-R functions, and the same device as the ridge penalty of Unit 2. <strong>Dropout</strong>, proposed by Srivastava and colleagues, takes a different route: during training, each hidden unit is set to zero at random with probability \(p\) on every forward pass, so the network cannot rely on any particular unit and learns redundant features; at test time all units are used, with the activations scaled by \(1 - p\).<sup class="c"><a href="#r27">[27]</a></sup> Dropout can be read as training an ensemble of thinned networks that share weights — the random-forest idea of Unit 4 inside one network. Neither `nnet` nor `neuralnet` implements dropout; the frameworks of Unit 8 do.

### 9.6 Vanishing gradients: why depth was hard

Section 8.4 observed that the gradient reaching the hidden layer was ten times smaller than the gradient at the output, because it had been multiplied by a weight and by a sigmoid derivative \(\leq 0.25\). In a network with ten hidden layers the factor is applied ten times, and the gradient of the first layer can be \(10^{-6}\) of the gradient of the last: the early layers, which should be learning the most basic features, receive almost no signal and barely move. Hochreiter identified this <strong>vanishing-gradient problem</strong> in 1991,<sup class="c"><a href="#r29">[29]</a></sup> and it is the main reason that networks with more than two or three hidden layers were impractical before about 2010. Three remedies broke the barrier: the ReLU activation, whose derivative is exactly 1 for active units;<sup class="c"><a href="#r26">[26]</a></sup> scaled initialisation (Section 9.3), which keeps the signal and gradient variance constant across layers;<sup class="c"><a href="#r25">[25]</a></sup> and, for very deep networks, architectural shortcuts (residual connections) that Unit 8 mentions. The experiment measures gradient magnitudes layer by layer in a network with ten hidden layers at initialisation.



In [ ]:
# Vanishing gradients: mean |dL/dW| per layer at initialisation, for a 9-[32 x 10]-1 network under three settings.
sizes_deep <- c(9, rep(32, 10), 1)
configs <- list("sigmoid, Glorot" = list("sigmoid", "glorot"),
                "sigmoid, N(0,0.1)" = list("sigmoid", 0.1),
                "ReLU, Glorot" = list("relu", "glorot"))
rows <- list()
for (nm in names(configs)) {
  cf <- configs[[nm]]; net <- mlp_init(sizes_deep, act = cf[[1]], init = cf[[2]], seed = 0)
  g <- mlp_backward(net, Ztr[1:64, ], ytr[1:64]); rows[[nm]] <- sapply(g$gW, function(G) mean(abs(G)))
}
cat("ratio of first-layer to last-layer gradient magnitude:\n")
for (nm in names(configs)) cat(sprintf("  %-18s: %.1e\n", nm, rows[[nm]][1] / rows[[nm]][11]))
plot(NA, xlim = c(1, 11), ylim = range(unlist(rows)[unlist(rows) > 0]), log = "y", xlab = "layer (1 = first hidden, 11 = output)",
     ylab = "mean |dL/dW| (log scale)", main = "Gradient magnitude by layer (11-layer network)")
cols <- c(ORANGE, GREY, BLUE)
for (k in seq_along(configs)) lines(1:11, rows[[k]], type = "o", pch = 19, cex = 0.6, col = cols[k])
legend("bottomright", names(configs), col = cols, lwd = 1, pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 14.</b> Mean absolute gradient of the weights of each layer, at initialisation, in a network with ten hidden layers of 32 units. With sigmoid units and Glorot initialisation (orange) the gradient shrinks by roughly a constant factor per layer and is smaller by several orders of magnitude at the first layer than at the output; with weights drawn from \(\mathcal{N}(0, 0.1)\) (grey) the shrinkage is faster still. With ReLU units (blue) the gradient stays within about one order of magnitude across all eleven layers. The sigmoid network's first layers would effectively not train; the ReLU network's would. The ratios printed above the figure make the comparison quantitative.</p>

<div class="box warn"><h4>Common misconception</h4><p>"More layers is always better." A deeper network has more capacity, but it is harder to optimise (vanishing or exploding gradients), needs more data, and on tabular problems of a few hundred cases rarely improves on one hidden layer. Depth pays when the input has compositional structure — pixels into edges into parts into objects — which is the subject of Unit 8. For the breast-cancer features, one hidden layer is the right default and Section 10 shows that even that is barely needed.</p></div>

### 9.7 Hyperparameters, and model selection with nnet

The choices above — architecture, activation, learning rate or optimiser, weight decay, epochs — are <strong>hyperparameters</strong>: quantities fixed before training that the training itself cannot set. They are selected by the methodology of Supplement S1: cross-validation on the training data, never on the test set. From here the notebook uses `nnet`, whose main hyperparameters are `size` (hidden units) and `decay` (the L2 strength). The first question is the size of the hidden layer, examined with a <em>validation curve</em>: cross-validated accuracy as a function of one hyperparameter.



In [ ]:
# Validation curve: 5-fold CV accuracy of a one-hidden-layer nnet as a function of the number of hidden units.
set.seed(0); feat_bc <- as.matrix(bc[, 1:9]); folds_bc <- sample(rep(1:5, length.out = nrow(feat_bc)))
cv_size <- function(h) {
  sapply(1:5, function(k) {
    trk <- folds_bc != k; m <- colMeans(feat_bc[trk, ]); s <- apply(feat_bc[trk, ], 2, sd)
    Xt <- scale(feat_bc[trk, ], m, s); Xv <- scale(feat_bc[!trk, ], m, s)
    fit <- nnet(Xt, y_mal[trk], size = h, entropy = TRUE, decay = 1e-4, maxit = 300, trace = FALSE, MaxNWts = 5000)
    mean((predict(fit, Xv) >= 0.5) == y_mal[!trk])
  })
}
hsizes <- c(1, 2, 4, 8, 16, 32); means <- sapply(hsizes, function(h) mean(cv_size(h))); sds <- sapply(hsizes, function(h) sd(cv_size(h)))
curve_tab <- data.frame(hidden_units = hsizes, CV_accuracy = round(means, 3), sd = round(sds, 3),
                        parameters = sapply(hsizes, function(h) n_parameters(c(9, h, 1))))
print(curve_tab, row.names = FALSE)
plot(hsizes, means, type = "o", pch = 19, col = BLUE, log = "x", ylim = c(0.93, 1), xlab = "hidden units",
     ylab = "5-fold CV accuracy", main = "Validation curve over the hidden-layer size")
arrows(hsizes, means - sds, hsizes, means + sds, angle = 90, code = 3, length = 0.03, col = BLUE)



<p class="caption"><b>Figure 15.</b> Cross-validated accuracy (with one standard deviation across folds) of the 9-\(h\)-1 network for \(h\) from 1 to 32. A single hidden unit — a logistic regression on one learned feature — already scores about 0.97, and the curve is essentially flat: within the fold-to-fold standard deviation, no size is clearly better than another. On these data the hidden-layer size is not a sensitive hyperparameter.</p>

### 9.8 Multi-class classification: the simulated glyphs

With \(K > 2\) classes the output layer has \(K\) softmax units and the loss is the categorical cross-entropy (Section 8.2); `nnet` does this when given a one-hot target matrix and `softmax = TRUE`. R has no handwritten-digits dataset, so the Python edition's `load_digits`<sup class="c"><a href="#r31">[31]</a></sup> is replaced by the simulated 8 × 8 glyphs of Section 4.3 — a five-class problem with a 64-dimensional input, the setting of Unit 8 in miniature.



In [ ]:
# A 64-32-5 softmax network on the simulated glyphs: cross-validated accuracy, then a look at the test errors.
set.seed(0); fdig <- sample(rep(1:5, length.out = nrow(Zd)))
cv_d <- sapply(1:5, function(k) {
  fit <- nnet(Zd[fdig != k, ], class.ind(dig$y[fdig != k]), size = 32, softmax = TRUE, maxit = 200, trace = FALSE, MaxNWts = 5000)
  mean((max.col(predict(fit, Zd[fdig == k, ])) - 1) == (as.integer(dig$y[fdig == k]) - 1))
})
set.seed(0); fit_d <- nnet(Zd_tr, Ytr, size = 32, softmax = TRUE, maxit = 300, trace = FALSE, MaxNWts = 5000)
pred_d <- max.col(predict(fit_d, Zd_te)) - 1; truth_d <- as.integer(yd_te) - 1; wrong <- which(pred_d != truth_d)
cat(sprintf("64-32-5 network: %d parameters | CV accuracy %.3f +/- %.3f | test accuracy %.3f (%d errors in %d images)\n",
            n_parameters(c(64, 32, 5)), mean(cv_d), sd(cv_d), mean(pred_d == truth_d), length(wrong), length(truth_d)))
labs <- names(templates)
par(mfrow = c(2, 8), mar = c(1.6, 0.5, 1.6, 0.5))
for (i in head(wrong, 16)) {
  image(matrix(dig$X[-tr_d, ][i, ], 8, 8)[, 8:1], col = grey.colors(10, 1, 0), axes = FALSE)
  title(sprintf("t:%s p:%s", labs[truth_d[i] + 1], labs[pred_d[i] + 1]), cex.main = 0.7)
}
if (length(wrong) < 16) for (i in seq_len(16 - length(wrong))) plot.new()



<p class="caption"><b>Figure 16.</b> The test glyphs that the 64-32-5 network misclassified, with the true (t) and predicted (p) class. Most are heavily corrupted by the simulated noise — the templates that share pixels (the filled box and the blob, the two bars) are the ones confused. The network's accuracy, well above 0.9 on held-out images, is in line with the k-NN and SVM results of Unit 3 on comparable data; with raw pixels as input, no method of the previous units has a decisive advantage, and the convolutional networks of Unit 8 are what push the error further down.</p>

### 9.9 Regression: a network versus linear regression

The same network with a linear output unit and the squared-error loss is a regression model (`nnet` with `linout = TRUE`). The Python edition used the sklearn diabetes data,<sup class="c"><a href="#r30">[30]</a></sup> which R does not ship; the R stand-in is the <strong>Boston housing</strong> data from `mlbench`<sup class="c"><a href="#r38">[38]</a></sup> — 506 census tracts, 13 variables, and the median house value to predict. The question of the Python edition stands: does a non-linear model do better than linear regression?



In [ ]:
# nnet (linear output) versus linear regression on Boston housing: 5-fold cross-validated R^2.
data(BostonHousing, package = "mlbench"); B <- BostonHousing; B$chas <- as.numeric(as.character(B$chas))
Xb13 <- as.matrix(B[, 1:13]); yb <- B$medv; set.seed(0); fb <- sample(rep(1:5, length.out = nrow(Xb13)))
R2 <- function(pred, obs) 1 - sum((obs - pred)^2) / sum((obs - mean(obs))^2)
run_cv <- function(fit_pred) sapply(1:5, function(k) {
  trk <- fb != k; m <- colMeans(Xb13[trk, ]); s <- apply(Xb13[trk, ], 2, sd)
  Xt <- scale(Xb13[trk, ], m, s); Xv <- scale(Xb13[!trk, ], m, s); R2(fit_pred(Xt, yb[trk], Xv), yb[!trk])
})
res_lm   <- run_cv(function(Xt, yt, Xv) { d <- data.frame(Xt, y = yt); predict(lm(y ~ ., d), data.frame(Xv)) })
res_n8   <- run_cv(function(Xt, yt, Xv) predict(nnet(Xt, yt, size = 8,  linout = TRUE, decay = 0.1, maxit = 500, trace = FALSE, MaxNWts = 3000), Xv))
res_n32  <- run_cv(function(Xt, yt, Xv) predict(nnet(Xt, yt, size = 32, linout = TRUE, decay = 0.1, maxit = 500, trace = FALSE, MaxNWts = 3000), Xv))
print(data.frame(model = c("linear regression (U2)", "nnet 13-8-1, decay 0.1", "nnet 13-32-1, decay 0.1"),
                 CV_R2 = round(c(mean(res_lm), mean(res_n8), mean(res_n32)), 3),
                 sd = round(c(sd(res_lm), sd(res_n8), sd(res_n32)), 3)), row.names = FALSE)



Here the honest answer differs from the Python edition's diabetes example, and the difference is instructive. On the Boston data a modest non-linear network <em>does</em> improve on linear regression — the cross-validated \(R^2\) rises from about 0.70 to roughly 0.8 — because the relationships between the predictors and house value are genuinely non-linear and 506 tracts are enough to reveal it. On the sklearn diabetes data, with 442 patients and a nearly linear signal, the network bought nothing. Same procedure, opposite conclusion: the methodological lesson is identical and is the point — run the comparison against the linear baseline, report what the data show, and do not assume in either direction.

## 10. Worked example: an MLP for breast-cancer diagnosis, end to end

Subunit 6.8 asks for a complete worked example. This section builds, selects, evaluates and interprets a neural network for the recurring diagnostic problem — malignant versus benign from the nine nuclear features<sup class="c"><a href="#r13">[13]</a></sup> — following the protocol of Supplement S1 at every step, and finishes by comparing it fairly with the methods of Units 3–4. The design decisions, made before looking at any result:

1. **Data split.** The 80/20 split of Section 8.6 (`tr`: 546 training, 137 test cases). The test cases are touched once, at the end.
2. **Pre-processing.** Standardisation fitted on the training folds only.
3. **Architecture and loss.** One hidden layer of logistic units, a sigmoid output, binary cross-entropy — `nnet` with `entropy = TRUE`.
4. **Regularisation.** Weight decay (`decay`), the `nnet` counterpart of the L2 penalty; `nnet` has no early-stopping option, so decay and the optimiser's own convergence do the regularising.
5. **Model selection.** A grid over hidden-layer size and decay, scored by 5-fold cross-validated ROC-AUC on the training set; AUC rather than accuracy because the operating threshold is a clinical decision made later.
6. **Evaluation.** On the test set: confusion matrix, sensitivity, specificity, AUC, and calibration.
7. **Interpretation.** Permutation importance: which features the network relies on.
8. **Comparison.** The same cross-validation applied to logistic regression, naive Bayes, SVM, k-NN, a decision tree and a random forest.



In [ ]:
# Step 5 - model selection by 5-fold cross-validated AUC over hidden sizes and decay.
set.seed(0); fg <- sample(rep(1:5, length.out = length(ytr)))
cv_auc <- function(size, decay) {
  mean(sapply(1:5, function(k) {
    trk <- fg != k; fit <- nnet(Ztr[trk, ], ytr[trk], size = size, entropy = TRUE, decay = decay,
                                maxit = 300, trace = FALSE, MaxNWts = 5000)
    as.numeric(auc(roc(ytr[!trk], as.vector(predict(fit, Ztr[!trk, ])), quiet = TRUE)))
  }))
}
grid <- expand.grid(size = c(4, 8, 16), decay = c(1e-4, 1e-2, 1))
grid$CV_AUC <- round(mapply(cv_auc, grid$size, grid$decay), 4)
print(reshape(grid, idvar = "size", timevar = "decay", direction = "wide"), row.names = FALSE)
best <- grid[which.max(grid$CV_AUC), ]
cat(sprintf("\nbest configuration: size = %d, decay = %g -> CV AUC %.4f\n", best$size, best$decay, best$CV_AUC))
set.seed(0); mlp_final <- nnet(Ztr, ytr, size = best$size, entropy = TRUE, decay = best$decay, maxit = 500, trace = FALSE, MaxNWts = 5000)



The table shows the flat landscape of Figure 15 again: every configuration reaches a cross-validated AUC of about 0.98–0.996, and the differences among them are of the order of the fold-to-fold standard deviation. A heavier decay (1) is in fact marginally the best on these easily separable data, but the margin is within the noise. `cv_auc` picks the best mean; it is refitted on all the training cases, and that single model goes to the test set. Unlike the Python edition, the grid has no early-stopping dimension, because `nnet` offers no early stopping — a library difference, not a change of method; weight decay plays the regularising role here.



In [ ]:
# Step 6 - evaluation on the held-out test cases: confusion matrix, sensitivity, specificity, AUC, calibration.
p_te <- as.vector(predict(mlp_final, Zte)); pred_te <- as.integer(p_te >= 0.5)
cm <- table(truth = yte, predicted = pred_te)
tn <- cm["0", "0"]; fp <- cm["0", "1"]; fn <- cm["1", "0"]; tp <- cm["1", "1"]
roc_mlp <- roc(yte, p_te, quiet = TRUE); auc_mlp <- as.numeric(auc(roc_mlp))
cat(sprintf("test set: TP %d, FN %d, TN %d, FP %d | accuracy %.3f, sensitivity %.3f, specificity %.3f, AUC %.4f\n",
            tp, fn, tn, fp, (tp + tn) / length(yte), tp / (tp + fn), tn / (tn + fp), auc_mlp))
co <- coords(roc_mlp, "all", ret = c("threshold", "sensitivity", "specificity"), transpose = FALSE)
i95 <- which(co$sensitivity >= 0.95); i95 <- i95[which.max(co$specificity[i95])]
cat(sprintf("threshold for sensitivity >= 0.95: %.3f -> sensitivity %.3f, specificity %.3f\n",
            co$threshold[i95], co$sensitivity[i95], co$specificity[i95]))
bins <- cut(p_te, quantile(p_te, seq(0, 1, 0.2)), include.lowest = TRUE)
cal <- data.frame(mean_pred = tapply(p_te, bins, mean), obs_frac = tapply(yte, bins, mean))
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
mosaicplot(cm, color = c(BLUE, ORANGE), main = "Confusion matrix (threshold 0.5)")
plot(1 - roc_mlp$specificities, roc_mlp$sensitivities, type = "l", col = BLUE, lwd = 1.6, xlab = "1 - specificity",
     ylab = "sensitivity", main = sprintf("ROC curve (AUC %.3f)", auc_mlp)); abline(0, 1, col = GREY, lty = 2)
points(1 - co$specificity[i95], co$sensitivity[i95], pch = 19, col = ORANGE)
plot(cal$mean_pred, cal$obs_frac, type = "o", pch = 19, col = BLUE, xlim = c(0, 1), ylim = c(0, 1),
     xlab = "mean predicted P(malignant)", ylab = "observed fraction", main = "Calibration"); abline(0, 1, col = GREY, lty = 2)



<p class="caption"><b>Figure 17.</b> Evaluation of the selected network on the held-out cases. Left: the confusion matrix at the default threshold of 0.5 — a few errors in each direction. Centre: the ROC curve, with an AUC near 0.99; the orange point is the lowest threshold at which at least 95 % of malignant cases are detected, the operating point a screening application would choose, a reminder that 0.5 is not a privileged threshold. Right: the calibration curve, observed fraction of malignant cases against mean predicted probability in five quantile bins — points near the diagonal mean that the network's probabilities can be read as probabilities. Networks trained with cross-entropy are generally reasonably well calibrated, in contrast with naive Bayes (Unit 3), whose posteriors are pushed to the extremes;<sup class="c"><a href="#r32">[32]</a></sup> the few test cases per bin make the curve noisy, however, and a calibration claim would need more data.</p>

Reading the numbers as an engineer would: the test set contains a few dozen malignant cases, so each false negative is worth more than two percentage points of sensitivity, and the confidence interval of any of these estimates is wide (Supplement S1). The honest summary is "AUC about 0.99, sensitivity about 0.90 and specificity about 0.98 at the default threshold — and a lower threshold reaches 100 % sensitivity at a small cost in specificity, on about 137 cases from one source", followed by the question of Unit 1: would it hold on aspirates from another laboratory?



In [ ]:
# Step 7 - interpretation by permutation importance: how much does the test AUC fall when one feature is shuffled?
set.seed(0); base_auc <- auc_mlp
imp <- sapply(1:9, function(j) {
  drops <- replicate(30, {
    Zp <- Zte; Zp[, j] <- sample(Zp[, j])
    base_auc - as.numeric(auc(roc(yte, as.vector(predict(mlp_final, Zp)), quiet = TRUE)))
  })
  c(mean = mean(drops), sd = sd(drops))
})
imp <- data.frame(feature = colnames(feat9), mean_AUC_drop = imp["mean", ], sd = imp["sd", ])
imp <- imp[order(-imp$mean_AUC_drop), ]
print(head(round(imp[, 2:3], 4)[order(-imp$mean_AUC_drop)[1:6], ], 4))
print(imp[1:6, ], row.names = FALSE, digits = 3)
top <- imp[6:1, ]
par(mar = c(4, 9, 2, 1)); barplot(top$mean_AUC_drop, names.arg = top$feature, horiz = TRUE, las = 1, col = BLUE,
        xlab = "decrease in test AUC when the feature is permuted", main = "Permutation importance of the selected MLP")



<p class="caption"><b>Figure 18.</b> The features whose random permutation most reduces the network's test AUC. A small number of nuclear-shape measurements carry almost all of the network's discriminative power, consistent with the pathology; the remaining features have an importance close to zero, either because the network does not need them or because their information is duplicated by correlated features (cell size, cell shape and clump thickness move together), in which case permuting one leaves the others to carry the signal. Permutation importance measures reliance, not causation, and shares the signal among correlated features arbitrarily.</p>

The final step is the comparison that every applied paper should contain and many omit: the same protocol applied to the simpler methods of Units 2–4.



In [ ]:
# Step 8 - the fair comparison: 5-fold CV AUC on the training set, then test AUC, for seven classifiers.
suppressPackageStartupMessages({ library(e1071); library(rpart); library(class); library(randomForest) })
score_test <- function(name, trainf) {
  cv <- sapply(1:5, function(k) trainf(Ztr[fg != k, ], ytr[fg != k], Ztr[fg == k, ], ytr[fg == k]))
  te <- trainf(Ztr, ytr, Zte, yte)
  c(CV_AUC = mean(cv), sd = sd(cv), test_AUC = te)
}
auc_of <- function(y, s) as.numeric(auc(roc(y, s, quiet = TRUE)))
models <- list(
  "logistic regression (U2)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(glm(yt ~ ., data.frame(Xt, yt = yt), family = binomial), data.frame(Xv), type = "response")),
  "Gaussian naive Bayes (U3)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(naiveBayes(Xt, factor(yt)), Xv, type = "raw")[, 2]),
  "SVM, RBF kernel (U3)" = function(Xt, yt, Xv, yv) auc_of(yv, attr(predict(svm(Xt, factor(yt), probability = TRUE), Xv, probability = TRUE), "probabilities")[, "1"]),
  "k-NN, k = 7 (U3)" = function(Xt, yt, Xv, yv) { pr <- knn(Xt, Xv, factor(yt), k = 7, prob = TRUE); pw <- attr(pr, "prob"); auc_of(yv, ifelse(pr == "1", pw, 1 - pw)) },
  "decision tree (U4)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(rpart(yt ~ ., data.frame(Xt, yt = factor(yt)), method = "class", control = rpart.control(maxdepth = 4)), data.frame(Xv), type = "prob")[, 2]),
  "random forest, 200 trees (U4)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(randomForest(Xt, factor(yt), ntree = 200), Xv, type = "prob")[, 2]),
  "MLP, selected above (U6)" = function(Xt, yt, Xv, yv) auc_of(yv, as.vector(predict(nnet(Xt, yt, size = best$size, entropy = TRUE, decay = best$decay, maxit = 500, trace = FALSE, MaxNWts = 5000), Xv))))
set.seed(0); comp <- t(sapply(names(models), function(nm) score_test(nm, models[[nm]])))
print(round(as.data.frame(comp), 3))



The table is the honest result of the worked example. Logistic regression, the SVM, the random forest and the network all reach a cross-validated AUC near or above 0.99 and the differences between them are within one standard deviation across folds; the single decision tree is the weakest, and naive Bayes is competitive on AUC though its probabilities are poorly calibrated, as Unit 3 showed. On a few hundred cases with nine designed features, the neural network's extra flexibility buys nothing that a regularised logistic regression does not already have — and the logistic regression has ten interpretable coefficients, trains in milliseconds and has no hyperparameters worth tuning. The network earns its place when the input is the image of the aspirate rather than nine numbers computed from it — the subject of Unit 8.

<div class="box key"><h4>What the worked example teaches</h4>
<ul>
<li>The protocol (split → standardise inside CV → cross-validated selection → single test evaluation → interpretation → comparison) is the same for every model in the course; the network adds hyperparameters, not steps.</li>
<li>Report AUC with its fold-to-fold variability, the operating point relevant to the clinical use, and calibration; a single accuracy is not an evaluation.</li>
<li>Always include the simple baselines. A result that does not beat logistic regression by more than the noise is a result in favour of logistic regression.</li>
</ul></div>

<div class="box key"><h4>Exercise 4 (preparation for Laboratory 4)</h4><p>Repeat the worked example on the simulated glyphs (five classes): select the hidden-layer size and <code>decay</code> by cross-validated accuracy, evaluate on the held-out split of Section 4.3 with a 5 × 5 confusion matrix, and compare with k-NN and a random forest. Then reduce the training set to 10 % of its size and repeat: which method degrades least, and why?</p></div>

## 11. From the multilayer perceptron to deep learning

### 11.1 Depth

A <strong>deep</strong> network is a multilayer perceptron with many hidden layers; there is no other difference of principle. The universal approximation theorem says one hidden layer suffices to <em>represent</em> any function, but it may need exponentially many units to do so, whereas a deep network can represent the same function with far fewer parameters by <em>composing</em> simple functions: the units of each layer re-use the features of the layer below, as a program re-uses subroutines. LeCun, Bengio and Hinton's review states the thesis of the field: deep networks learn a hierarchy of representations, each level more abstract than the last, and this hierarchy is what the hand-designed features of classical pattern recognition could never provide.<sup class="c"><a href="#r6">[6]</a></sup> What made depth practical after decades of failure was the conjunction of Section 9's remedies — ReLU, scaled initialisation, dropout, Adam — with two external factors: labelled datasets of millions of examples, and graphics processing units (GPUs), whose thousands of simple cores execute the matrix products of forward and backward propagation in parallel, a hundred times faster than a general-purpose processor. The 2012 ImageNet network was trained on two GPUs for about a week.<sup class="c"><a href="#r5">[5]</a></sup>

### 11.2 Convolution: a preview of Unit 8

A fully connected first layer on a 256 × 256 image has 65,536 inputs per unit, and a hidden layer of 1,000 units has 65 million weights — too many to learn from any medical dataset, and wasteful, because the feature that detects a tumour margin in the upper left corner is the same feature that detects it in the lower right. A <strong>convolutional layer</strong> solves both problems: each unit looks only at a small window of the image (say 3 × 3 pixels), and all units of the layer share the same weights, sliding the window across the image. The layer then has 9 weights instead of 65 million, detects its feature wherever it occurs, and stacks of such layers learn the hierarchy edges → textures → parts → objects. LeCun introduced the architecture for handwritten-digit recognition in the late 1980s and gave it its definitive form, LeNet-5, in 1998;<sup class="c"><a href="#r33">[33]</a></sup> every medical-imaging system of Section 5 is a descendant. The chunk below applies two hand-designed 3 × 3 filters to a glyph to show what a convolutional unit computes; in Unit 8 the filters are learned by backpropagation exactly as the weights of this unit were.



In [ ]:
# What a convolutional unit computes: a 3x3 filter slid over an image (two hand-designed Sobel edge detectors).
conv2 <- function(img, k) {
  kk <- (nrow(k) - 1) / 2; pad <- matrix(0, nrow(img) + 2 * kk, ncol(img) + 2 * kk)
  pad[(kk + 1):(kk + nrow(img)), (kk + 1):(kk + ncol(img))] <- img; out <- matrix(0, nrow(img), ncol(img))
  for (i in seq_len(nrow(img))) for (j in seq_len(ncol(img))) out[i, j] <- sum(pad[i:(i + 2 * kk), j:(j + 2 * kk)] * k)
  out
}
img <- templates$box                                                           # an 8 x 8 glyph
filters <- list("vertical edges" = matrix(c(-1, 0, 1, -2, 0, 2, -1, 0, 1), 3, byrow = TRUE),
                "horizontal edges" = matrix(c(-1, -2, -1, 0, 0, 0, 1, 2, 1), 3, byrow = TRUE))
cat(sprintf("a 3x3 convolutional unit has %d parameters whatever the image size; a fully connected unit on this 8x8 image has %d, on a 256x256 image %d\n",
            3 * 3 + 1, 8 * 8 + 1, 256 * 256 + 1))
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
image(img[, 8:1], col = grey.colors(10, 1, 0), axes = FALSE, main = "input glyph (8x8)")
for (nm in names(filters)) { r <- conv2(img, filters[[nm]]); image(r[, 8:1], col = hcl.colors(20, "Blue-Red"), axes = FALSE, main = paste("response to", nm)) }



<p class="caption"><b>Figure 19.</b> A simulated glyph (a box outline) and its responses to two 3 × 3 filters (Sobel operators), each applied at every position of the image. The left response map lights up where intensity changes from left to right — the vertical sides of the box — and the right one where it changes from top to bottom — the horizontal sides. Each map is the output of one convolutional unit, with nine weights, evaluated at 64 positions. A convolutional network learns dozens of such filters in its first layer from the data, then combines their responses in the next.</p>

### 11.3 Sequences and attention

Two further architectures complete the map. <strong>Recurrent networks</strong> (Section 3.1) process a sequence one element at a time, carrying a hidden state forward, and were the standard model for signals and language through the 2010s; the <em>long short-term memory</em> unit of Hochreiter and Schmidhuber solved their vanishing-gradient problem across time with gated memory cells,<sup class="c"><a href="#r34">[34]</a></sup> and recurrent networks read electrocardiograms, electroencephalograms and clinical time series. The <strong>Transformer</strong> of 2017 replaced recurrence by <em>attention</em>, a mechanism in which every element of the sequence computes a weighted sum over all the others, with weights that are themselves learned functions of the content; because the computation over positions is parallel rather than sequential, Transformers train efficiently on very long sequences and very large corpora,<sup class="c"><a href="#r35">[35]</a></sup> and they underlie the large language models and the protein-structure predictors of Unit 1. Both are, at the level of this unit, arrangements of weighted sums and activation functions trained by backpropagation; what changes is the pattern of connections and of weight sharing.

<div class="box bridge"><h4>Bridge to Unit 8</h4><p>Unit 8 trains a convolutional network for the classification of medical image patterns. Everything it needs is in this unit: the layer as a matrix product, the softmax output and cross-entropy loss, backpropagation, the learning rate and early stopping, standardisation of the input, and the evaluation protocol of Section 10. The convolutional layer of Figure 19 is the one new component.</p></div>

## 12. Summary

You should now be able to explain, without notes:

- The brain in numbers — about 86 billion neurons, \(10^{14}\)–\(10^{15}\) synapses, millisecond components, massive parallelism — and the five properties that artificial networks borrow (simple units, weighted connections, distributed representation, graceful degradation, learning by weight change), together with the honest statement that an artificial network is inspired by, not a model of, the brain.
- The components of a unit — inputs, weights, bias, weighted sum, activation function, output — and of a network — input, hidden and output layers, architecture, feed-forward versus recurrent — and how to count the parameters of an architecture.
- Why a single sigmoid neuron is logistic regression, and why a network with hidden layers is a logistic regression on learned features.
- The universal approximation theorem (what it promises and what it does not), and the four limitations: data-hungry, opaque, many hyperparameters, non-convex training.
- The activation functions step, sigmoid, tanh, ReLU and softmax, their derivatives, and why the derivative matters (saturation, dead units, vanishing gradients).
- How one unit computes AND and OR, the geometric and algebraic proof that none computes XOR, and why that motivates hidden layers.
- Rosenblatt's learning rule, its derivation as a step along \(y\mathbf{x}\), the convergence theorem and its \((R/\gamma)^2\) bound, the hand trace on AND, the behaviour on non-separable data, and the Adaline's delta rule as gradient descent on a smooth loss.
- Forward propagation as matrix products; the cross-entropy and squared-error losses and why cross-entropy is preferred for classification; backpropagation as the chain rule organised around \(\delta^{(l)}\), with the four matrix equations; the hand trace of one step on the 2-2-1 network; gradient checking.
- The practical choices — learning rate, epochs and early stopping, initialisation, standardisation, weight decay and dropout — and the experiments that exhibit crawling, divergence, overfitting and vanishing gradients.
- The protocol of the worked example and its result: on tabular data of a few hundred cases a network matches, but does not beat, logistic regression and random forests; its advantage lies in raw inputs — images, signals, sequences — which Unit 8 takes up.
- What depth, convolution, recurrence and attention add to the multilayer perceptron, and why GPUs matter.

## 13. Self-assessment

<details><summary>1. A colleague says "our network has three layers: input, hidden and output". How many layers of computing units does it have, and why does the count matter when reading a paper?</summary>Two: the input layer holds the features and computes nothing. Papers and software differ on whether they count it, so an architecture should be given as the list of layer sizes (e.g. 9-16-1), which is unambiguous; the parameter count, which bounds the data needed, follows from that list.</details>

<details><summary>2. Count the parameters of a 64-32-5 network, and state what fraction of the 750 simulated glyphs that number represents.</summary>\(64 \times 32 + 32 + 32 \times 5 + 5 = 2048 + 32 + 160 + 5 = 2245\) parameters — about three parameters per training image. Such a network can memorise the training set; cross-validation and weight decay are what keep it honest.</details>

<details><summary>3. By hand: a unit has weights \(w = (2, -1)\), bias \(b = -0.5\) and a sigmoid activation. Compute its output for \(\mathbf{x} = (1, 1)\) and for \(\mathbf{x} = (0, 1)\). Which side of the boundary is each on?</summary>\(z = 2 - 1 - 0.5 = 0.5 \Rightarrow a = \sigma(0.5) \approx 0.622\) for \((1,1)\); \(z = 0 - 1 - 0.5 = -1.5 \Rightarrow a = \sigma(-1.5) \approx 0.182\) for \((0,1)\). The first is on the positive side of the line \(2x_1 - x_2 = 0.5\) (output above 0.5), the second on the negative side.</details>

<details><summary>4. Give weights and bias for a step-activation unit that computes NOR\((x_1, x_2)\), and explain in one sentence why no choice computes XOR.</summary>\(w = (-1, -1)\), \(b = 0.5\): \(z = 0.5\) for \((0,0)\) and negative otherwise. XOR requires the two diagonal corners of the unit square to lie on one side of a line and the other two on the other side, which no line achieves (adding the inequalities for \((0,1)\) and \((1,0)\) and subtracting the one for \((0,0)\) contradicts the one for \((1,1)\)).</details>

<details><summary>5. In the perceptron hand trace of Section 7.3, the second epoch begins with \(\mathbf{w} = (1,1)\), \(b = 0\). Compute the weights after the first two steps of epoch 2 (inputs \((0,0)\) then \((0,1)\), \(\eta = 1\)).</summary>Step 1: \(z = 0 \Rightarrow \hat{y} = +1 \neq -1\); update \(b \leftarrow -1\), \(\mathbf{w}\) unchanged \((1,1)\). Step 2: \(z = 1 - 1 = 0 \Rightarrow \hat{y} = +1 \neq -1\); update \(\mathbf{w} \leftarrow (1,1) - (0,1) = (1,0)\), \(b \leftarrow -2\). This matches the rows printed by the code.</details>

<details><summary>6. The perceptron convergence theorem bounds the number of mistakes by \((R/\gamma)^2\). What happens to the bound if every feature is multiplied by 10, and what does that tell you about the data, not the algorithm?</summary>Nothing: \(R\) and \(\gamma\) are both multiplied by 10 and the ratio is unchanged. The bound depends on the <em>geometry</em> of the problem — how large the margin is relative to the spread of the data — not on its units; a problem whose classes nearly touch (small \(\gamma/R\)) needs many updates whatever the learning rate.</details>

<details><summary>7. Why is the step function unusable as a hidden activation in a network trained by backpropagation, and why did the sigmoid later give way to the ReLU?</summary>Its derivative is zero everywhere, so \(\delta^{(l)} = (\dots) \odot \varphi'(z)\) is zero and no weight receives a gradient. The sigmoid is differentiable but its derivative is at most 0.25 and vanishes when the unit saturates, so gradients shrink geometrically with depth; the ReLU's derivative is exactly 1 for active units, so gradients pass through undiminished (Figure 14).</details>

<details><summary>8. In the backpropagation hand trace, the hidden-layer gradients were about ten times smaller than the output-layer gradients. Identify the two factors responsible and say which one the ReLU removes.</summary>\(\delta^{(1)}_j = \delta^{(2)} W^{(2)}_j\, \sigma'(z^{(1)}_j)\): the error is multiplied by the connecting weight (magnitude 0.6 or 0.4) and by the sigmoid derivative (at most 0.25). The ReLU replaces the second factor by 1 for active units; the first factor is controlled by initialisation.</details>

<details><summary>9. Read the code: in <code>mlp_backward</code>, why is <code>delta</code> initialised as <code>(A[[length(A)]] - y) / n</code> with no activation derivative, and what would have to change if the loss were the squared error?</summary>For a sigmoid output with the binary cross-entropy, \(\partial L/\partial z^{(L)} = (\hat{y} - y)/n\) exactly — the \(\sigma'(z)\) factor cancels against the derivative of the log. With the squared error the line would become <code>(A[[length(A)]] - y) * A[[length(A)]] * (1 - A[[length(A)]]) / n</code>, as in the XOR chunk, and training would slow down whenever the output saturates.</details>

<details><summary>10. Figure 13 shows the validation loss rising after a number of epochs while the training loss keeps falling. Name the phenomenon, a remedy, and say why <code>nnet</code> does not implement early stopping.</summary>Overfitting. Remedies: early stopping (keep the weights of the best validation epoch), weight decay (<code>decay</code>), a smaller network, or more data. <code>nnet</code> uses a batch quasi-Newton optimiser (BFGS) that runs to <code>maxit</code> or its tolerance rather than epoch-by-epoch SGD, so there is no natural place to monitor a validation set and stop; the from-scratch loop, which steps epoch by epoch, can.</details>

<details><summary>11. In the worked example the network, the SVM, the random forest and logistic regression all reached a cross-validated AUC near 0.99. A reviewer asks you to justify using the network. What do you answer?</summary>That on these data the network is not justified: it does not beat the simplest baseline by more than the fold-to-fold variability, and it is less interpretable and more expensive to tune. The justification for networks lies in problems with raw, high-dimensional inputs (images, signals) where features must be learned; the designed features of this dataset already contain the information.</details>

<details><summary>12. Section 9.4 standardised inside the cross-validation. What error would fitting the scaling on the full dataset before cross-validation introduce, and would it make the reported accuracy higher or lower than the truth?</summary>Data leakage: the test folds' means and standard deviations would influence the training folds' standardisation, giving the model information about the test data. The reported accuracy would be optimistically biased — higher than the accuracy on genuinely new cases (Supplement S1).</details>

## Glossary

<dl class="gloss">
<dt>Activation function</dt><dd>The fixed non-linear function \(\varphi\) applied to a unit's weighted sum; step, sigmoid, tanh, ReLU for hidden or output units, softmax for a multi-class output layer.</dd>
<dt>Adaline / delta rule</dt><dd>Widrow and Hoff's linear unit trained by gradient descent on the squared error; its update \(\mathbf{w} \leftarrow \mathbf{w} + \eta (y - z)\mathbf{x}\) is the one-layer case of backpropagation.</dd>
<dt>Architecture</dt><dd>The specification of a network: number and size of layers, activation functions, pattern of connections (fully connected, convolutional, recurrent).</dd>
<dt>Backpropagation</dt><dd>The algorithm that computes the gradient of the loss with respect to every weight by applying the chain rule layer by layer from the output backwards, reusing \(\delta^{(l)} = \partial L / \partial \mathbf{z}^{(l)}\).</dd>
<dt>Batch, mini-batch, epoch</dt><dd>The set of cases used for one gradient step (all cases: full batch; a subset: mini-batch; one case: online); one pass through the whole training set is an epoch.</dd>
<dt>Bias</dt><dd>The constant term \(b\) added to a unit's weighted sum; equivalently a weight on a constant input of 1. It shifts the decision boundary away from the origin.</dd>
<dt>Convolutional layer</dt><dd>A layer whose units each see a small window of the input and share the same weights, so that a feature is detected wherever it occurs; the building block of image networks (Unit 8).</dd>
<dt>Cross-entropy</dt><dd>The negative log-likelihood loss for probabilistic classification: binary for a sigmoid output, categorical for a softmax output. With these outputs its gradient with respect to the logits is \(\hat{y} - y\).</dd>
<dt>Deep learning</dt><dd>Neural networks with many hidden layers, trained end to end from raw inputs; the learned hierarchy of representations is its defining feature.</dd>
<dt>Dropout</dt><dd>A regularisation method that sets each hidden unit to zero at random during training, forcing redundant representations.</dd>
<dt>Early stopping</dt><dd>Ending training when the loss on a validation set stops improving, and keeping the weights of the best epoch; the most common regulariser for networks.</dd>
<dt>Feed-forward / recurrent</dt><dd>A network without cycles, whose output depends only on the current input / a network with feedback connections, whose output depends on the sequence of past inputs.</dd>
<dt>Glorot (Xavier) initialisation</dt><dd>Random initial weights with variance \(2/(n_{\text{in}} + n_{\text{out}})\), chosen so that signal and gradient variance stay roughly constant across layers.</dd>
<dt>Gradient checking</dt><dd>Verifying a backpropagation implementation by comparing its derivatives with centred finite differences of the loss.</dd>
<dt>Hebbian learning</dt><dd>Hebb's 1949 rule: a synapse is strengthened when the pre- and postsynaptic neurons are active together; the first proposal that learning is the adjustment of connection strengths.</dd>
<dt>Hidden layer</dt><dd>A layer whose activations are neither inputs nor outputs; its units compute the learned features on which the output layer operates.</dd>
<dt>Hyperparameter</dt><dd>A quantity fixed before training (architecture, learning rate, penalty strength, epochs) and chosen by cross-validation, as opposed to a parameter learned by gradient descent.</dd>
<dt>Learning rate</dt><dd>The step size \(\eta\) of gradient descent; too small crawls, too large diverges.</dd>
<dt>Linearly separable</dt><dd>A labelled sample for which some hyperplane puts all cases of one class on one side and all of the other on the other; the condition under which the perceptron converges.</dd>
<dt>Logit</dt><dd>The pre-activation of an output unit, \(z = \mathbf{w}^\top \mathbf{a} + b\), before the sigmoid or softmax; equal to the log-odds for a sigmoid output.</dd>
<dt>McCulloch–Pitts neuron</dt><dd>The 1943 unit: binary inputs, a threshold, a binary output; the first formal model of a neuron as a logical device.</dd>
<dt>Multilayer perceptron (MLP)</dt><dd>A fully connected feed-forward network with one or more hidden layers and differentiable activations, trained by backpropagation.</dd>
<dt>Perceptron</dt><dd>Rosenblatt's threshold unit and its learning rule: update \(\mathbf{w} \leftarrow \mathbf{w} + \eta y \mathbf{x}\) on misclassified examples only.</dd>
<dt>Perceptron convergence theorem</dt><dd>Novikoff's result: on linearly separable data with margin \(\gamma\) and radius \(R\) the perceptron makes at most \((R/\gamma)^2\) mistakes.</dd>
<dt>ReLU</dt><dd>Rectified linear unit, \(\varphi(z) = \max(0, z)\); the default hidden activation of deep networks because its derivative is 1 for active units.</dd>
<dt>Saturation</dt><dd>The regime in which a sigmoid or tanh unit's pre-activation is large in magnitude, its output is at an extreme and its derivative is near zero, so little gradient passes through it.</dd>
<dt>Sigmoid</dt><dd>The logistic function \(\sigma(z) = 1/(1 + e^{-z})\) (R's <code>plogis</code>); the output activation for binary classification, giving \(P(y = 1 \mid \mathbf{x})\).</dd>
<dt>Softmax</dt><dd>The function that maps \(K\) logits to \(K\) positive numbers summing to 1; the output activation for multi-class classification.</dd>
<dt>Universal approximation theorem</dt><dd>Cybenko's and Hornik's result that a network with one hidden layer of enough non-linear units can approximate any continuous function on a bounded domain; an existence result that says nothing about learnability.</dd>
<dt>Vanishing gradient</dt><dd>The geometric shrinkage of gradients as they are propagated back through many saturating layers (or time steps), so that early layers barely learn; mitigated by ReLU, scaled initialisation and architectural shortcuts.</dd>
<dt>Weight</dt><dd>The adjustable real number on a connection, multiplying the signal that passes along it; the parameters of a network are its weights and biases.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">McCulloch WS, Pitts W. A logical calculus of the ideas immanent in nervous activity. <em>Bull Math Biophys</em>. 1943;5:115–133. <a href="https://doi.org/10.1007/BF02478259">doi:10.1007/BF02478259</a></li>
<li id="r2">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386–408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r3">Minsky M, Papert S. <em>Perceptrons: An Introduction to Computational Geometry</em>. MIT Press; 1969. Reissued 2017. <a href="https://doi.org/10.7551/mitpress/11301.001.0001">doi:10.7551/mitpress/11301.001.0001</a></li>
<li id="r4">Rumelhart DE, Hinton GE, Williams RJ. Learning representations by back-propagating errors. <em>Nature</em>. 1986;323:533–536. <a href="https://doi.org/10.1038/323533a0">doi:10.1038/323533a0</a></li>
<li id="r5">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84–90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r6">LeCun Y, Bengio Y, Hinton G. Deep learning. <em>Nature</em>. 2015;521:436–444. <a href="https://doi.org/10.1038/nature14539">doi:10.1038/nature14539</a></li>
<li id="r7">Azevedo FAC, Carvalho LRB, Grinberg LT, et al. Equal numbers of neuronal and nonneuronal cells make the human brain an isometrically scaled-up primate brain. <em>J Comp Neurol</em>. 2009;513(5):532–541. <a href="https://doi.org/10.1002/cne.21974">doi:10.1002/cne.21974</a></li>
<li id="r8">Herculano-Houzel S. The human brain in numbers: a linearly scaled-up primate brain. <em>Front Hum Neurosci</em>. 2009;3:31. <a href="https://doi.org/10.3389/neuro.09.031.2009">doi:10.3389/neuro.09.031.2009</a></li>
<li id="r9">Pakkenberg B, Pelvig D, Marner L, et al. Aging and the human neocortex. <em>Exp Gerontol</em>. 2003;38(1–2):95–99. <a href="https://doi.org/10.1016/S0531-5565(02)00151-1">doi:10.1016/S0531-5565(02)00151-1</a></li>
<li id="r10">Hodgkin AL, Huxley AF. A quantitative description of membrane current and its application to conduction and excitation in nerve. <em>J Physiol</em>. 1952;117(4):500–544. <a href="https://doi.org/10.1113/jphysiol.1952.sp004764">doi:10.1113/jphysiol.1952.sp004764</a></li>
<li id="r11">Brunel N, van Rossum MCW. Lapicque's 1907 paper: from frogs to integrate-and-fire. <em>Biol Cybern</em>. 2007;97:337–339. <a href="https://doi.org/10.1007/s00422-007-0190-0">doi:10.1007/s00422-007-0190-0</a></li>
<li id="r12">Hebb DO. <em>The Organization of Behavior: A Neuropsychological Theory</em>. New York: Wiley; 1949. Reissued Psychology Press; 2002. <a href="https://doi.org/10.4324/9781410612403">doi:10.4324/9781410612403</a></li>
<li id="r13">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r14">Cybenko G. Approximation by superpositions of a sigmoidal function. <em>Math Control Signals Syst</em>. 1989;2:303–314. <a href="https://doi.org/10.1007/BF02551274">doi:10.1007/BF02551274</a></li>
<li id="r15">Hornik K, Stinchcombe M, White H. Multilayer feedforward networks are universal approximators. <em>Neural Netw</em>. 1989;2(5):359–366. <a href="https://doi.org/10.1016/0893-6080(89)90020-8">doi:10.1016/0893-6080(89)90020-8</a></li>
<li id="r16">Baker JA, Kornguth PJ, Lo JY, Williford ME, Floyd CE. Breast cancer: prediction with artificial neural network based on BI-RADS standardized lexicon. <em>Radiology</em>. 1995;196(3):817–822. <a href="https://doi.org/10.1148/radiology.196.3.7644649">doi:10.1148/radiology.196.3.7644649</a></li>
<li id="r17">Lisboa PJ, Taktak AFG. The use of artificial neural networks in decision support in cancer: a systematic review. <em>Neural Netw</em>. 2006;19(4):408–415. <a href="https://doi.org/10.1016/j.neunet.2005.10.007">doi:10.1016/j.neunet.2005.10.007</a></li>
<li id="r18">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402–2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r19">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115–118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r20">De Fauw J, Ledsam JR, Romera-Paredes B, et al. Clinically applicable deep learning for diagnosis and referral in retinal disease. <em>Nat Med</em>. 2018;24:1342–1350. <a href="https://doi.org/10.1038/s41591-018-0107-6">doi:10.1038/s41591-018-0107-6</a></li>
<li id="r21">Hannun AY, Rajpurkar P, Haghpanahi M, et al. Cardiologist-level arrhythmia detection and classification in ambulatory electrocardiograms using a deep neural network. <em>Nat Med</em>. 2019;25:65–69. <a href="https://doi.org/10.1038/s41591-018-0268-3">doi:10.1038/s41591-018-0268-3</a></li>
<li id="r22">Rajkomar A, Dean J, Kohane I. Machine learning in medicine. <em>N Engl J Med</em>. 2019;380(14):1347–1358. <a href="https://doi.org/10.1056/NEJMra1814259">doi:10.1056/NEJMra1814259</a></li>
<li id="r23">Novikoff AB. On convergence proofs on perceptrons. In: <em>Proceedings of the Symposium on the Mathematical Theory of Automata</em>, vol. XII. Brooklyn, NY: Polytechnic Institute of Brooklyn; 1962:615–622. (Conference proceedings without a DOI; the theorem and proof are reproduced in Minsky and Papert [3] and in Bishop's <em>Neural Networks for Pattern Recognition</em>, listed under Textbooks.)</li>
<li id="r24">Widrow B, Hoff ME. Adaptive switching circuits. <em>1960 IRE WESCON Convention Record</em>, part 4. New York: IRE; 1960:96–104. Stanford Electronics Laboratories Technical Report 1553-1. <a href="https://doi.org/10.21236/AD0241531">doi:10.21236/AD0241531</a></li>
<li id="r25">Glorot X, Bengio Y. Understanding the difficulty of training deep feedforward neural networks. <em>Proceedings of the 13th International Conference on Artificial Intelligence and Statistics (AISTATS)</em>, PMLR 9:249–256; 2010. <a href="https://proceedings.mlr.press/v9/glorot10a.html">proceedings.mlr.press/v9/glorot10a.html</a> (PMLR proceedings carry no DOI)</li>
<li id="r26">Nair V, Hinton GE. Rectified linear units improve restricted Boltzmann machines. <em>Proceedings of the 27th International Conference on Machine Learning (ICML)</em>; 2010:807–814. <a href="https://icml.cc/Conferences/2010/papers/432.pdf">icml.cc/Conferences/2010/papers/432.pdf</a> (ICML 2010 proceedings carry no DOI)</li>
<li id="r27">Srivastava N, Hinton G, Krizhevsky A, Sutskever I, Salakhutdinov R. Dropout: a simple way to prevent neural networks from overfitting. <em>J Mach Learn Res</em>. 2014;15(56):1929–1958. <a href="https://jmlr.org/papers/v15/srivastava14a.html">jmlr.org/papers/v15/srivastava14a.html</a> (JMLR articles carry no DOI)</li>
<li id="r28">Kingma DP, Ba J. Adam: a method for stochastic optimization. <em>3rd International Conference on Learning Representations (ICLR)</em>; 2015. <a href="https://doi.org/10.48550/arXiv.1412.6980">doi:10.48550/arXiv.1412.6980</a></li>
<li id="r29">Hochreiter S. The vanishing gradient problem during learning recurrent neural nets and problem solutions. <em>Int J Uncertain Fuzziness Knowl-Based Syst</em>. 1998;6(2):107–116. <a href="https://doi.org/10.1142/S0218488598000094">doi:10.1142/S0218488598000094</a></li>
<li id="r30">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. (Source of the sklearn diabetes dataset used in the Python edition.) <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r31">Alpaydin E, Kaynak C. Cascading classifiers. <em>Kybernetika</em>. 1998;34(4):369–374. UCI Optical Recognition of Handwritten Digits dataset (used in the Python edition; simulated in R). <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r32">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. <em>Proceedings of the 22nd International Conference on Machine Learning (ICML)</em>; 2005:625–632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r33">LeCun Y, Bottou L, Bengio Y, Haffner P. Gradient-based learning applied to document recognition. <em>Proc IEEE</em>. 1998;86(11):2278–2324. <a href="https://doi.org/10.1109/5.726791">doi:10.1109/5.726791</a></li>
<li id="r34">Hochreiter S, Schmidhuber J. Long short-term memory. <em>Neural Comput</em>. 1997;9(8):1735–1780. <a href="https://doi.org/10.1162/neco.1997.9.8.1735">doi:10.1162/neco.1997.9.8.1735</a></li>
<li id="r35">Vaswani A, Shazeer N, Parmar N, et al. Attention is all you need. <em>Advances in Neural Information Processing Systems 30 (NeurIPS 2017)</em>. <a href="https://doi.org/10.48550/arXiv.1706.03762">doi:10.48550/arXiv.1706.03762</a></li>
<li id="r36">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. (The reference for <code>nnet</code>, <code>MASS</code> and the R modelling functions used here.) <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r37">Fritsch S, Guenther F, Wright MN. neuralnet: Training of Neural Networks. R package. <a href="https://doi.org/10.32614/CRAN.package.neuralnet">doi:10.32614/CRAN.package.neuralnet</a></li>
<li id="r38">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package (source of the Wisconsin breast-cancer and Boston housing data used here). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. (The R companion to the whole course; free PDF at statlearning.com.) <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Bishop CM. <em>Neural Networks for Pattern Recognition</em>. Oxford University Press; 1995. — The classical treatment of the multilayer perceptron, backpropagation and its statistical interpretation.</li>
<li>Goodfellow I, Bengio Y, Courville A. <em>Deep Learning</em>. MIT Press; 2016. — Chapters 6–8 expand Sections 8–9; free online at deeplearningbook.org.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 11 (neural networks) for the statistician's view. Free PDF.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS)<sup class="c"><a href="#r36">[36]</a></sup> — the reference for <code>nnet</code> and the R modelling functions; and James et al., <em>Introduction to Statistical Learning</em><sup class="c"><a href="#r39">[39]</a></sup> for the R companion to the course.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The <code>nnet</code> and <code>neuralnet</code><sup class="c"><a href="#r37">[37]</a></sup> package manuals on CRAN — documentation of the training functions used from Section 9 onward.</li>
<li>Nielsen M. <em>Neural Networks and Deep Learning</em> (online book) — <a href="http://neuralnetworksanddeeplearning.com">neuralnetworksanddeeplearning.com</a>; a from-scratch exposition of backpropagation in the spirit of Section 8.</li>
</ul>
